In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os, glob
import pandas as pd

# --- 1. locate the dataset (don't trust a hardcoded path) ---
candidates = glob.glob("/kaggle/input/**/MINDsmall_train", recursive=True)
if not candidates:
    raise FileNotFoundError("MINDsmall_train not found under /kaggle/input")
TRAIN_DIR = candidates[0]
DEV_DIR   = TRAIN_DIR.replace("MINDsmall_train", "MINDsmall_dev")
print("train dir:", TRAIN_DIR)
print("dev dir:  ", DEV_DIR, "(exists:", os.path.isdir(DEV_DIR), ")")
print("files:", os.listdir(TRAIN_DIR))

# --- 2. MIND tsv files have NO header row. These are the official columns. ---
NEWS_COLS = ["news_id", "category", "subcategory",
             "title", "abstract", "url",
             "title_entities", "abstract_entities"]

BEHAVIORS_COLS = ["impression_id", "user_id", "time",
                  "history", "impressions"]

news = pd.read_csv(os.path.join(TRAIN_DIR, "news.tsv"),
                   sep="\t", header=None, names=NEWS_COLS,
                   quoting=3)  # quoting=3 = QUOTE_NONE, tsv has raw quotes

beh  = pd.read_csv(os.path.join(TRAIN_DIR, "behaviors.tsv"),
                   sep="\t", header=None, names=BEHAVIORS_COLS,
                   quoting=3)

# --- 3. look at shape ---
print("\n=== news.tsv ===")
print("rows:", len(news), "| cols:", list(news.columns))
print(news.head(3).to_string())

print("\n=== behaviors.tsv ===")
print("rows:", len(beh), "| cols:", list(beh.columns))
print(beh.head(3).to_string())

# --- 4. inspect the tricky fields ---
print("\n--- one behaviors row, decoded ---")
row = beh.iloc[0]
print("user_id      :", row.user_id)
print("time         :", row.time)
print("history (raw):", row.history)
print("impressions  :", row.impressions)

# history is space-separated news IDs; impressions are 'NewsID-label'
hist = str(row.history).split()
imps = [x.split("-") for x in str(row.impressions).split()]
print("history parsed:", hist[:5], "...", f"({len(hist)} clicked previously)")
print("impressions parsed (first 5):", imps[:5])

# --- 5. missing-value + entity peek ---
print("\n--- nulls in news ---")
print(news.isnull().sum())
print("\n--- one title_entities value ---")
print(news["title_entities"].iloc[0][:300])

In [ ]:
import glob
print(glob.glob("/kaggle/input/**/MINDsmall_dev*", recursive=True))
print(glob.glob("/kaggle/input/**/*dev*", recursive=True))

In [ ]:
import glob, os
import polars as pl   # EB-NeRD ships parquet with array columns; polars handles them cleanly

# 1. locate the demo bundle (don't hardcode the mount slug)
demo = glob.glob("/kaggle/input/**/ebnerd_demo", recursive=True)
print("demo root:", demo)
DEMO = demo[0]

# 2. confirm validation actually has its two files (the collapsed folder in your screenshot)
print("\ntrain/     :", os.listdir(f"{DEMO}/train"))
print("validation/:", os.listdir(f"{DEMO}/validation"))
print("root       :", os.listdir(DEMO))

# 3. peek at each table's schema + one row
def peek(path, n=3):
    df = pl.read_parquet(path)
    print(f"\n=== {path.split('/')[-1]} ===")
    print("rows:", df.height, "| cols:", df.columns)
    print(df.head(n))

peek(f"{DEMO}/articles.parquet")
peek(f"{DEMO}/train/behaviors.parquet")
peek(f"{DEMO}/train/history.parquet")

In [ ]:
import polars as pl, glob
DEMO = glob.glob("/kaggle/input/**/ebnerd_demo", recursive=True)[0]

beh = pl.read_parquet(f"{DEMO}/train/behaviors.parquet")

# a) confirm inview vs clicked are lists, and clicked ⊆ inview
row = beh.select(["article_ids_inview","article_ids_clicked"]).row(1)
print("inview :", row[0][:8], "...")
print("clicked:", row[1])

# b) how many candidates per impression, and clicks per impression?
print("\ninview len stats:")
print(beh.select(pl.col("article_ids_inview").list.len().alias("n")).describe())
print("clicked len stats:")
print(beh.select(pl.col("article_ids_clicked").list.len().alias("n")).describe())

In [ ]:
import os, zipfile
from kaggle_secrets import UserSecretsClient
from huggingface_hub import hf_hub_download

# --- 1. pull token from Kaggle Secrets (Add-ons → Secrets) ---
HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")   # <- your secret's name

# --- 2. download both splits into /kaggle/working ---
REPO = "yjw1029/MIND"
DEST = "/kaggle/working/mind"
os.makedirs(DEST, exist_ok=True)

def get_split(zip_name):
    out_dir = os.path.join(DEST, zip_name.replace(".zip", ""))
    if os.path.isdir(out_dir) and os.listdir(out_dir):
        print(f"skip (already have) {out_dir}")
        return out_dir
    zip_path = hf_hub_download(
        repo_id=REPO, filename=zip_name, repo_type="dataset",
        local_dir=DEST, token=HF_TOKEN,
    )
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(out_dir)
    os.remove(zip_path)                    # drop the zip, keep the tsvs
    print(f"extracted -> {out_dir}: {os.listdir(out_dir)}")
    return out_dir

train_dir = get_split("MINDsmall_train.zip")
dev_dir   = get_split("MINDsmall_dev.zip")

# --- 3. verify ---
for d in (train_dir, dev_dir):
    print(d, "->", os.listdir(d))

In [ ]:
import pandas as pd
DEV = "/kaggle/input/datasets/wrathofgod123/mind-dev/MINDsmall_dev"

beh = pd.read_csv(f"{DEV}/behaviors.tsv", sep="\t", header=None,
                  names=["impression_id","user_id","time","history","impressions"],
                  quoting=3)
news = pd.read_csv(f"{DEV}/news.tsv", sep="\t", header=None,
                   names=["news_id","category","subcategory","title","abstract",
                          "url","title_entities","abstract_entities"], quoting=3)

print("dev behaviors rows:", len(beh), "| dev news rows:", len(news))
print("sample impressions:", beh['impressions'].iloc[0])
# labels present? count clicks vs skips in first few rows
labels = [x.split("-")[1] for imp in beh['impressions'].head(100) for x in str(imp).split()]
from collections import Counter
print("label distribution (first 100 impressions):", Counter(labels))

In [ ]:
import glob, os

def find_one(pattern, must_contain=None):
    hits = sorted(glob.glob(pattern, recursive=True))
    if must_contain:
        hits = [h for h in hits if os.path.exists(os.path.join(h, must_contain))]
    return hits[0] if hits else None

PATHS = {
    # --- required (small + demo) ---
    "mind_train":   find_one("/kaggle/input/**/MINDsmall_train", "news.tsv"),
    "mind_dev":     find_one("/kaggle/input/**/MINDsmall_dev",   "news.tsv"),
    "ebnerd_demo":  find_one("/kaggle/input/**/ebnerd_demo",     "articles.parquet"),
    "ebnerd_small": find_one("/kaggle/input/**/ebnerd_small",    "articles.parquet"),
    # --- optional (large) ---
    "mind_large_train": find_one("/kaggle/input/**/MINDlarge_train", "news.tsv"),
    "mind_large_dev":   find_one("/kaggle/input/**/MINDlarge_dev",   "news.tsv"),
    "mind_large_test":  find_one("/kaggle/input/**/MINDlarge_test",  "news.tsv"),
}

print("Resolved paths:")
for k, v in PATHS.items():
    print(f"  {'✓' if v else '·'}  {k:18s} -> {v}")

# only small + demo are required to proceed
required = ["mind_train", "mind_dev", "ebnerd_demo"]
missing = [k for k in required if not PATHS[k]]
assert not missing, f"Missing REQUIRED splits: {missing}"
print("\nRequired splits present. Large is optional and won't block the pipeline.")

# quick label check on large_test — confirm it's the blanked hidden test
lt = PATHS["mind_large_test"]
if lt:
    import pandas as pd
    b = pd.read_csv(f"{lt}/behaviors.tsv", sep="\t", header=None,
                    names=["impression_id","user_id","time","history","impressions"],
                    quoting=3, nrows=200)
    from collections import Counter
    labs = [x.split("-")[-1] for imp in b["impressions"] for x in str(imp).split() if "-" in str(x)]
    print("\nlarge_test label sample:", Counter(labs))
    # if this is all '0' or has no labels, it's the hidden test (submission-only)

In [ ]:
import os, pandas as pd
from collections import Counter

LT = "/kaggle/input/datasets/wrathofgod123/mind-large/mindlargetest/MINDlarge_test"

print("files:", os.listdir(LT))

news = pd.read_csv(f"{LT}/news.tsv", sep="\t", header=None,
                   names=["news_id","category","subcategory","title","abstract",
                          "url","title_entities","abstract_entities"], quoting=3)
beh  = pd.read_csv(f"{LT}/behaviors.tsv", sep="\t", header=None,
                   names=["impression_id","user_id","time","history","impressions"],
                   quoting=3)

print("news rows:", len(news), "| behaviors rows:", len(beh))
print("\nsample impression:", beh['impressions'].iloc[0][:200])

# check whether labels exist
tokens = [x for imp in beh['impressions'].head(200) for x in str(imp).split()]
has_dash = [t for t in tokens if "-" in t]
print("\ntokens have '-label' suffix?", len(has_dash) > 0)
if has_dash:
    print("label sample:", Counter(t.split("-")[-1] for t in has_dash))
else:
    print("no labels — bare candidate IDs (confirms hidden test)")
    print("first tokens:", tokens[:8])

In [1]:
import polars as pl, glob
DEMO = glob.glob("/kaggle/input/**/ebnerd_demo", recursive=True)[0]
v = pl.read_parquet(f"{DEMO}/validation/behaviors.parquet")
print("val behaviors cols:", v.columns)
print("val rows:", v.height)
print("clicked populated?", v.select(pl.col("article_ids_clicked").list.len().mean()).item())

val behaviors cols: ['impression_id', 'article_id', 'impression_time', 'read_time', 'scroll_percentage', 'device_type', 'article_ids_inview', 'article_ids_clicked', 'user_id', 'is_sso_user', 'gender', 'postcode', 'age', 'is_subscriber', 'session_id', 'next_read_time', 'next_scroll_percentage']
val rows: 25356
clicked populated? 1.005876321186307


In [2]:
import polars as pl, glob

PREFIX = "ebnerd"
def _prefix(col): return pl.concat_str([pl.lit(f"{PREFIX}:"), col.cast(pl.Utf8)])

DEMO = glob.glob("/kaggle/input/**/ebnerd_demo", recursive=True)[0]

# ---- articles (bundle root, shared across splits) ----
a = pl.read_parquet(f"{DEMO}/articles.parquet")
articles = a.select(
    article_id=_prefix(pl.col("article_id")),
    dataset=pl.lit(PREFIX),
    title=pl.col("title").fill_null(""),
    abstract=pl.col("subtitle").fill_null(""),
    body=pl.col("body").fill_null(""),
    category=pl.col("category_str").fill_null(""),
    entities=pl.col("ner_clusters"),
    published_time=pl.col("published_time"),
)

# ---- impressions (per split) ----
b = pl.read_parquet(f"{DEMO}/train/behaviors.parquet")
impressions = b.select(
    impression_id=pl.col("impression_id"),
    dataset=pl.lit(PREFIX),
    user_id=_prefix(pl.col("user_id")),
    timestamp=pl.col("impression_time"),
    candidate_ids=pl.col("article_ids_inview").list.eval(_prefix(pl.element())),
    labels=pl.col("article_ids_clicked").list.eval(_prefix(pl.element())),
    session_id=pl.col("session_id"),
)

# ---- history (per split, separate table) ----
h = pl.read_parquet(f"{DEMO}/train/history.parquet")
history = h.select(
    user_id=_prefix(pl.col("user_id")),
    dataset=pl.lit(PREFIX),
    article_ids=pl.col("article_id_fixed").list.eval(_prefix(pl.element())),
    timestamps=pl.col("impression_time_fixed"),
)

for name, df in [("articles", articles), ("impressions", impressions), ("history", history)]:
    print(f"\n=== {name} === rows={df.height}, cols={df.columns}")
    print(df.head(3))


=== articles === rows=11777, cols=['article_id', 'dataset', 'title', 'abstract', 'body', 'category', 'entities', 'published_time']
shape: (3, 8)
┌────────────┬─────────┬────────────┬────────────┬────────────┬────────────┬───────────┬───────────┐
│ article_id ┆ dataset ┆ title      ┆ abstract   ┆ body       ┆ category   ┆ entities  ┆ published │
│ ---        ┆ ---     ┆ ---        ┆ ---        ┆ ---        ┆ ---        ┆ ---       ┆ _time     │
│ str        ┆ str     ┆ str        ┆ str        ┆ str        ┆ str        ┆ list[str] ┆ ---       │
│            ┆         ┆            ┆            ┆            ┆            ┆           ┆ datetime[ │
│            ┆         ┆            ┆            ┆            ┆            ┆           ┆ μs]       │
╞════════════╪═════════╪════════════╪════════════╪════════════╪════════════╪═══════════╪═══════════╡
│ ebnerd:303 ┆ ebnerd  ┆ Ishockey-s ┆ ISHOCKEY:  ┆ Ambitioner ┆ sport      ┆ []        ┆ 2003-08-2 │
│ 7230       ┆         ┆ piller:    ┆ Ishockey

In [3]:
import polars as pl, glob, datetime as dt

PREFIX = "ebnerd"
def _prefix(col): return pl.concat_str([pl.lit(f"{PREFIX}:"), col.cast(pl.Utf8)])
DEMO = glob.glob("/kaggle/input/**/ebnerd_demo", recursive=True)[0]

# --- parse train impressions + history (from earlier) ---
b = pl.read_parquet(f"{DEMO}/train/behaviors.parquet")
imps = b.select(
    impression_id="impression_id", dataset=pl.lit(PREFIX),
    user_id=_prefix(pl.col("user_id")), timestamp=pl.col("impression_time"),
    candidate_ids=pl.col("article_ids_inview").list.eval(_prefix(pl.element())),
    labels=pl.col("article_ids_clicked").list.eval(_prefix(pl.element())),
    session_id="session_id",
)
h = pl.read_parquet(f"{DEMO}/train/history.parquet")
hist = h.select(
    user_id=_prefix(pl.col("user_id")), dataset=pl.lit(PREFIX),
    article_ids=pl.col("article_id_fixed").list.eval(_prefix(pl.element())),
    timestamps=pl.col("impression_time_fixed"),
)

# --- 1. temporal train/val split (last 1 day of train -> val) ---
tmax = imps.select(pl.col("timestamp").max()).item()
cutoff = tmax - dt.timedelta(days=1)
train_imps = imps.filter(pl.col("timestamp") <= cutoff)
val_imps   = imps.filter(pl.col("timestamp") >  cutoff)
print(f"train time range: {imps.select(pl.col('timestamp').min()).item()} .. {tmax}")
print(f"cutoff: {cutoff}")
print(f"train impressions: {train_imps.height} | val impressions: {val_imps.height}")

# --- 2. no-future-leakage check (Q9) ---
hist_ts = (hist.select("user_id","timestamps")
               .explode("timestamps").rename({"timestamps":"hist_ts"}))
joined = imps.select("user_id","timestamp").join(hist_ts, on="user_id", how="inner")
violations = joined.filter(pl.col("hist_ts") >= pl.col("timestamp"))
print(f"\nleakage violations: {violations.height}")
assert violations.height == 0, f"LEAKAGE FOUND:\n{violations.head(3)}"
print("no-future-leakage assertion PASSED")

train time range: 2023-05-18 07:00:03 .. 2023-05-25 06:59:52
cutoff: 2023-05-24 06:59:52
train impressions: 21318 | val impressions: 3406

leakage violations: 0
no-future-leakage assertion PASSED


In [5]:
!pip install rank_bm25 -q

In [6]:


import re, numpy as np, polars as pl, glob
from rank_bm25 import BM25Okapi

PREFIX="ebnerd"
def _prefix(c): return pl.concat_str([pl.lit(f"{PREFIX}:"), c.cast(pl.Utf8)])
_WORD=re.compile(r"[^\W\d_]+",re.UNICODE)
def tok(t): return _WORD.findall(t.lower()) if t else []

DEMO=glob.glob("/kaggle/input/**/ebnerd_demo",recursive=True)[0]

# ---- parse (articles + train impressions + history) ----
a=pl.read_parquet(f"{DEMO}/articles.parquet")
articles=a.select(article_id=_prefix(pl.col("article_id")),
    title=pl.col("title").fill_null(""), abstract=pl.col("subtitle").fill_null(""),
    body=pl.col("body").fill_null(""), published_time="published_time")
b=pl.read_parquet(f"{DEMO}/train/behaviors.parquet")
imps=b.select(impression_id="impression_id", user_id=_prefix(pl.col("user_id")),
    timestamp="impression_time",
    candidate_ids=pl.col("article_ids_inview").list.eval(_prefix(pl.element())),
    labels=pl.col("article_ids_clicked").list.eval(_prefix(pl.element())))
h=pl.read_parquet(f"{DEMO}/train/history.parquet")
hist=h.select(user_id=_prefix(pl.col("user_id")),
    article_ids=pl.col("article_id_fixed").list.eval(_prefix(pl.element())),
    timestamps="impression_time_fixed")

# ---- build index (title + abstract) ----
ids, texts = [], []
for aid,ti,ab in articles.select(["article_id","title","abstract"]).iter_rows():
    ids.append(aid); texts.append(tok(f"{ti} {ab}"))
bm25=BM25Okapi(texts); ids_arr=np.array(ids)
title_lut={aid:tok(ti) for aid,ti in articles.select(["article_id","title"]).iter_rows()}
pub_lut={aid:pt for aid,pt in articles.select(["article_id","published_time"]).iter_rows()}
hist_lut={u:(ai or [],ts or []) for u,ai,ts in hist.select(["user_id","article_ids","timestamps"]).iter_rows()}

# ---- evaluate recall@K on a sample (demo has 21k train imps; sample for speed) ----
Ks=(50,100,200); maxK=200; MAX_HIST=30
sample=imps.sample(n=min(2000,imps.height),seed=0)
hits={k:0 for k in Ks}; scored=0
for imp in sample.iter_rows(named=True):
    labs=imp["labels"]
    if not labs: continue
    ai,ts=hist_lut.get(imp["user_id"],([],[]))
    if not ai: continue
    if imp["timestamp"] and ts and len(ts)==len(ai):
        kept=[(x,t) for x,t in zip(ai,ts) if t is not None and t<imp["timestamp"]]
        kept.sort(key=lambda z:z[1]); ai=[x for x,_ in kept][-MAX_HIST:]
    else: ai=ai[-MAX_HIST:]
    q=[]; [q.extend(title_lut.get(x,[])) for x in ai]
    if not q: continue
    sc=bm25.get_scores(q)
    tsi=imp["timestamp"]
    mask=np.array([(pub_lut.get(a) is None) or (pub_lut.get(a)<tsi) for a in ids])
    sc=np.where(mask,sc,-np.inf)
    kk=min(maxK,len(sc))
    idx=np.argpartition(-sc,kk-1)[:kk]; idx=idx[np.argsort(-sc[idx])]
    ranked=ids_arr[idx]; ls=set(labs); scored+=1
    for k in Ks:
        if ls & set(ranked[:k].tolist()): hits[k]+=1

print("scored:",scored)
for k in Ks: print(f"recall@{k}: {hits[k]/scored:.4f}")

scored: 2000
recall@50: 0.0170
recall@100: 0.0275
recall@200: 0.0460


In [7]:
!pip install bm25s -q

import re, numpy as np, polars as pl, glob
import bm25s

PREFIX="ebnerd"
def _prefix(c): return pl.concat_str([pl.lit(f"{PREFIX}:"), c.cast(pl.Utf8)])
_WORD=re.compile(r"[^\W\d_]+",re.UNICODE)
def tok(t): return _WORD.findall(t.lower()) if t else []

DEMO=glob.glob("/kaggle/input/**/ebnerd_demo",recursive=True)[0]

# ---- parse ----
a=pl.read_parquet(f"{DEMO}/articles.parquet")
articles=a.select(article_id=_prefix(pl.col("article_id")),
    title=pl.col("title").fill_null(""), abstract=pl.col("subtitle").fill_null(""),
    published_time="published_time")
b=pl.read_parquet(f"{DEMO}/train/behaviors.parquet")
imps=b.select(impression_id="impression_id", user_id=_prefix(pl.col("user_id")),
    timestamp="impression_time",
    labels=pl.col("article_ids_clicked").list.eval(_prefix(pl.element())))
h=pl.read_parquet(f"{DEMO}/train/history.parquet")
hist=h.select(user_id=_prefix(pl.col("user_id")),
    article_ids=pl.col("article_id_fixed").list.eval(_prefix(pl.element())),
    timestamps="impression_time_fixed")

# ---- build bm25s index (pre-tokenized, Danish-safe) ----
ids, corpus = [], []
for aid,ti,ab in articles.select(["article_id","title","abstract"]).iter_rows():
    ids.append(aid); corpus.append(tok(f"{ti} {ab}"))
ids_arr=np.array(ids)
id_to_row={a:i for i,a in enumerate(ids)}

retriever=bm25s.BM25()
retriever.index(corpus)                      # fast, vectorized

title_lut={aid:tok(ti) for aid,ti in articles.select(["article_id","title"]).iter_rows()}
pub_arr=np.array([articles.select("published_time").row(id_to_row[a])[0] for a in ids])  # aligned
hist_lut={u:(ai or [],ts or []) for u,ai,ts in
          hist.select(["user_id","article_ids","timestamps"]).iter_rows()}

# ---- build all queries first (with leakage guard) ----
MAX_HIST=30
q_tokens, q_labels, q_ts = [], [], []
for imp in imps.iter_rows(named=True):
    labs=imp["labels"]
    if not labs: continue
    ai,ts=hist_lut.get(imp["user_id"],([],[]))
    if not ai: continue
    if imp["timestamp"] and ts and len(ts)==len(ai):
        kept=[(x,t) for x,t in zip(ai,ts) if t is not None and t<imp["timestamp"]]
        kept.sort(key=lambda z:z[1]); ai=[x for x,_ in kept][-MAX_HIST:]
    else: ai=ai[-MAX_HIST:]
    q=[]; [q.extend(title_lut.get(x,[])) for x in ai]
    if not q: continue
    q_tokens.append(q); q_labels.append(set(labs)); q_ts.append(imp["timestamp"])

print("queries to score:", len(q_tokens))

# ---- batch retrieve top-K for ALL queries at once ----
Ks=(50,100,200); maxK=200
res, scores = retriever.retrieve(q_tokens, k=maxK, show_progress=True)  # res: [n, maxK] row indices

# ---- recall@K with freshness filter ----
hits={k:0 for k in Ks}
for i in range(len(q_tokens)):
    rows=res[i]                              # article row-indices, ranked
    ts=q_ts[i]
    # freshness: keep only articles published before impression
    if ts is not None:
        rows=[r for r in rows if pub_arr[r] is None or pub_arr[r]<ts]
    ranked=ids_arr[rows]
    for k in Ks:
        if q_labels[i] & set(ranked[:k].tolist()): hits[k]+=1

n=len(q_tokens)
for k in Ks: print(f"recall@{k}: {hits[k]/n:.4f}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.7/74.7 kB 1.6 MB/s eta 0:00:00ta 0:00:01


BM25S Create Vocab:   0%|          | 0/11777 [00:00<?, ?it/s]

BM25S Convert tokens to indices:   0%|          | 0/11777 [00:00<?, ?it/s]

BM25S Count Tokens:   0%|          | 0/11777 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/11777 [00:00<?, ?it/s]

queries to score: 24724


BM25S Retrieve:   0%|          | 0/24724 [00:00<?, ?it/s]

recall@50: 0.0145
recall@100: 0.0268
recall@200: 0.0383


In [10]:
import math, numpy as np
from collections import Counter

# ============ 1. BM25 slate scorer (self-contained) ============
class BM25SlateScorer:
    def __init__(self, corpus_tokens, k1=1.5, b=0.75):
        self.k1,self.b=k1,b; self.N=len(corpus_tokens)
        self.doc_tf=[Counter(d) for d in corpus_tokens]
        self.doc_len=np.array([len(d) for d in corpus_tokens],dtype=float)
        self.avgdl=self.doc_len.mean() if self.N else 0.0
        df=Counter()
        for tf in self.doc_tf: df.update(tf.keys())
        self.idf={t:math.log((self.N-d+0.5)/(d+0.5)+1.0) for t,d in df.items()}
    def score(self,q,row):
        if not q: return 0.0
        tf=self.doc_tf[row]; dl=self.doc_len[row]
        dn=self.k1*(1-self.b+self.b*dl/self.avgdl); s=0.0
        for t in set(q):
            f=tf.get(t,0)
            if f: s+=self.idf.get(t,0.0)*(f*(self.k1+1))/(f+dn)
        return s

slate=BM25SlateScorer(corpus)   # `corpus` = tokenized articles from Q2 cell

# ============ 2. metrics ============
def auc_imp(sc,lb):
    pos=lb==1; neg=lb==0; npos,nneg=pos.sum(),neg.sum()
    if npos==0 or nneg==0: return None
    order=np.argsort(sc); ranks=np.empty_like(order,dtype=float)
    ranks[order]=np.arange(1,len(sc)+1)
    return float((ranks[pos].sum()-npos*(npos+1)/2)/(npos*nneg))
def mrr_imp(sc,lb):
    rl=lb[np.argsort(-sc)]; idx=np.nonzero(rl==1)[0]
    return 1.0/(idx[0]+1) if len(idx) else 0.0
def dcg(r): return np.sum(r/np.log2(np.arange(2,len(r)+2)))
def ndcg_imp(sc,lb,k):
    rk=lb[np.argsort(-sc)][:k]; idl=np.sort(lb)[::-1][:k]
    i=dcg(idl); return float(dcg(rk)/i) if i>0 else 0.0

# ============ 3. build records with scores ============
rec_k=5; Ks=(5,10)
rows=[]; recommended=set()
total_pop=sum(pop.values())
for imp in imps_full.iter_rows(named=True):
    labs=set(imp["labels"] or []); cand=imp["candidate_ids"]
    if not labs or not cand: continue
    y=np.array([1 if c in labs else 0 for c in cand])
    if y.sum()==0: continue
    q=hist_query(imp["user_id"],imp["timestamp"])
    crows=[id_to_row.get(c,-1) for c in cand]
    sc=np.array([slate.score(q,r) if r>=0 else 0.0 for r in crows])
    d={"auc":auc_imp(sc,y),"mrr":mrr_imp(sc,y)}
    for k in Ks: d[f"ndcg@{k}"]=ndcg_imp(sc,y,k)
    top=np.array(cand)[np.argsort(-sc)[:rec_k]]; recommended.update(top.tolist())
    cats=[cat_lut.get(a) for a in top]
    if len(cats)>1:
        same=sum(1 for i in range(len(cats)) for j in range(i+1,len(cats))
            if cats[i] is not None and cats[i]==cats[j])
        d["diversity"]=1.0-same/(len(cats)*(len(cats)-1)/2)
    novs=[-math.log2(pop.get(a,1)/total_pop) for a in top if total_pop]
    d["novelty"]=float(np.mean(novs)) if novs else 0.0
    d["_slice"]="cold" if hlen.get(imp["user_id"],0)<5 else "warm"
    rows.append(d)

# ============ 4. aggregate + bootstrap 95% CI ============
mkeys=["auc","mrr","ndcg@5","ndcg@10","diversity","novelty"]
def agg(subset,n_boot=1000,seed=0):
    rng=np.random.default_rng(seed); out={}
    for m in mkeys:
        vals=np.array([r[m] for r in subset if r.get(m) is not None],dtype=float)
        if len(vals)==0: out[m]=(np.nan,np.nan,np.nan); continue
        boots=np.array([rng.choice(vals,len(vals),replace=True).mean() for _ in range(n_boot)])
        out[m]=(vals.mean(),*np.percentile(boots,[2.5,97.5]))
    return out

print(f"eval impressions: {len(rows)}\n")
for name,sub in [("OVERALL",rows),
                 ("COLD",[r for r in rows if r['_slice']=='cold']),
                 ("WARM",[r for r in rows if r['_slice']=='warm'])]:
    if not sub: continue
    print(f"--- {name} (n={len(sub)}) ---")
    for m,(p,lo,hi) in agg(sub).items():
        print(f"  {m:10s} {p:.4f}  [{lo:.4f}, {hi:.4f}]")
    print()
print(f"coverage: {len(recommended)/len(ids):.4f}  ({len(recommended)}/{len(ids)} articles)")

eval impressions: 24724

--- OVERALL (n=24724) ---
  auc        0.5188  [0.5147, 0.5229]
  mrr        0.3390  [0.3354, 0.3425]
  ndcg@5     0.3733  [0.3689, 0.3777]
  ndcg@10    0.4560  [0.4528, 0.4596]
  diversity  0.7885  [0.7863, 0.7910]
  novelty    10.3117  [10.2987, 10.3235]

--- WARM (n=24724) ---
  auc        0.5188  [0.5147, 0.5229]
  mrr        0.3390  [0.3354, 0.3425]
  ndcg@5     0.3733  [0.3689, 0.3777]
  ndcg@10    0.4560  [0.4528, 0.4596]
  diversity  0.7885  [0.7863, 0.7910]
  novelty    10.3117  [10.2987, 10.3235]

coverage: 0.1766  (2080/11777 articles)


In [11]:
import math, numpy as np
from collections import Counter

# ---- 0. re-parse impressions WITH candidate slates ----
b = pl.read_parquet(f"{DEMO}/train/behaviors.parquet")
imps_full = b.select(
    impression_id="impression_id", user_id=_prefix(pl.col("user_id")),
    timestamp="impression_time",
    candidate_ids=pl.col("article_ids_inview").list.eval(_prefix(pl.element())),
    labels=pl.col("article_ids_clicked").list.eval(_prefix(pl.element())),
)

cat_lut = {aid: c for aid, c in a.select(
    [_prefix(pl.col("article_id")).alias("id"),
     pl.col("category_str").fill_null("")]).iter_rows()}

pop = Counter()
for (labs,) in imps_full.select("labels").iter_rows():
    for x in (labs or []): pop[x] += 1
total_pop = sum(pop.values())

# ---- head = top 20% most-clicked, tail = rest ----
sorted_by_pop = sorted(pop.items(), key=lambda kv: -kv[1])
n_head = max(1, int(0.20 * len(sorted_by_pop)))
head_set = {aid for aid, _ in sorted_by_pop[:n_head]}
print(f"head articles: {len(head_set)} | tail: {len(sorted_by_pop)-len(head_set)}\n")

# ---- 1. BM25 slate scorer ----
class BM25SlateScorer:
    def __init__(self, corpus_tokens, k1=1.5, b=0.75):
        self.k1, self.b = k1, b; self.N = len(corpus_tokens)
        self.doc_tf = [Counter(d) for d in corpus_tokens]
        self.doc_len = np.array([len(d) for d in corpus_tokens], dtype=float)
        self.avgdl = self.doc_len.mean() if self.N else 0.0
        df = Counter()
        for tf in self.doc_tf: df.update(tf.keys())
        self.idf = {t: math.log((self.N-d+0.5)/(d+0.5)+1.0) for t, d in df.items()}
    def score(self, q, row):
        if not q: return 0.0
        tf = self.doc_tf[row]; dl = self.doc_len[row]
        dn = self.k1*(1-self.b+self.b*dl/self.avgdl); s = 0.0
        for t in set(q):
            f = tf.get(t, 0)
            if f: s += self.idf.get(t, 0.0)*(f*(self.k1+1))/(f+dn)
        return s
slate = BM25SlateScorer(corpus)

# ---- 2. metrics ----
def auc_imp(sc, lb):
    pos = lb == 1; neg = lb == 0; npos, nneg = pos.sum(), neg.sum()
    if npos == 0 or nneg == 0: return None
    order = np.argsort(sc); ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(sc)+1)
    return float((ranks[pos].sum()-npos*(npos+1)/2)/(npos*nneg))
def mrr_imp(sc, lb):
    rl = lb[np.argsort(-sc)]; idx = np.nonzero(rl == 1)[0]
    return 1.0/(idx[0]+1) if len(idx) else 0.0
def dcg(r): return np.sum(r/np.log2(np.arange(2, len(r)+2)))
def ndcg_imp(sc, lb, k):
    rk = lb[np.argsort(-sc)][:k]; idl = np.sort(lb)[::-1][:k]
    i = dcg(idl); return float(dcg(rk)/i) if i > 0 else 0.0

def hist_query(uid, ts):
    ai, tss = hist_lut.get(uid, ([], []))
    if not ai: return []
    if ts and tss and len(tss) == len(ai):
        kept = [(x, t) for x, t in zip(ai, tss) if t is not None and t < ts]
        kept.sort(key=lambda z: z[1]); ai = [x for x, _ in kept][-30:]
    else:
        ai = ai[-30:]
    q = []
    for x in ai: q.extend(title_lut.get(x, []))
    return q

# ---- 3. build records ----
rec_k = 5; Ks = (5, 10)
rows = []; recommended = set()
for imp in imps_full.iter_rows(named=True):
    labs = set(imp["labels"] or []); cand = imp["candidate_ids"]
    if not labs or not cand: continue
    y = np.array([1 if c in labs else 0 for c in cand])
    if y.sum() == 0: continue
    q = hist_query(imp["user_id"], imp["timestamp"])
    crows = [id_to_row.get(c, -1) for c in cand]
    sc = np.array([slate.score(q, r) if r >= 0 else 0.0 for r in crows])
    d = {"auc": auc_imp(sc, y), "mrr": mrr_imp(sc, y)}
    for k in Ks: d[f"ndcg@{k}"] = ndcg_imp(sc, y, k)
    top = np.array(cand)[np.argsort(-sc)[:rec_k]]
    recommended.update(top.tolist())
    cats = [cat_lut.get(x) for x in top]
    if len(cats) > 1:
        same = sum(1 for i in range(len(cats)) for j in range(i+1, len(cats))
                   if cats[i] is not None and cats[i] == cats[j])
        d["diversity"] = 1.0 - same/(len(cats)*(len(cats)-1)/2)
    novs = [-math.log2(pop.get(x, 1)/total_pop) for x in top if total_pop]
    d["novelty"] = float(np.mean(novs)) if novs else 0.0
    d["_slice"] = "head" if (labs & head_set) else "tail"
    rows.append(d)

# ---- 4. aggregate + bootstrap CI ----
mkeys = ["auc", "mrr", "ndcg@5", "ndcg@10", "diversity", "novelty"]
def agg(subset, n_boot=1000, seed=0):
    rng = np.random.default_rng(seed); out = {}
    for m in mkeys:
        vals = np.array([r[m] for r in subset if r.get(m) is not None], dtype=float)
        if len(vals) == 0: out[m] = (np.nan, np.nan, np.nan); continue
        boots = np.array([rng.choice(vals, len(vals), replace=True).mean()
                          for _ in range(n_boot)])
        out[m] = (vals.mean(), *np.percentile(boots, [2.5, 97.5]))
    return out

print(f"eval impressions: {len(rows)}\n")
for name, sub in [("OVERALL", rows),
                  ("HEAD", [r for r in rows if r['_slice'] == 'head']),
                  ("TAIL", [r for r in rows if r['_slice'] == 'tail'])]:
    if not sub: continue
    print(f"--- {name} (n={len(sub)}) ---")
    for m, (p, lo, hi) in agg(sub).items():
        print(f"  {m:10s} {p:.4f}  [{lo:.4f}, {hi:.4f}]")
    print()
print(f"coverage: {len(recommended)/len(ids):.4f}  ({len(recommended)}/{len(ids)} articles)")

head articles: 222 | tail: 892

eval impressions: 24724

--- OVERALL (n=24724) ---
  auc        0.5188  [0.5147, 0.5229]
  mrr        0.3390  [0.3354, 0.3425]
  ndcg@5     0.3733  [0.3689, 0.3777]
  ndcg@10    0.4560  [0.4528, 0.4596]
  diversity  0.7885  [0.7863, 0.7910]
  novelty    10.3117  [10.2987, 10.3235]

--- HEAD (n=12019) ---
  auc        0.5307  [0.5249, 0.5368]
  mrr        0.3490  [0.3435, 0.3543]
  ndcg@5     0.3845  [0.3782, 0.3904]
  ndcg@10    0.4660  [0.4607, 0.4707]
  diversity  0.7902  [0.7874, 0.7933]
  novelty    10.1601  [10.1418, 10.1783]

--- TAIL (n=12705) ---
  auc        0.5076  [0.5016, 0.5132]
  mrr        0.3295  [0.3246, 0.3349]
  ndcg@5     0.3626  [0.3570, 0.3687]
  ndcg@10    0.4466  [0.4416, 0.4517]
  diversity  0.7870  [0.7837, 0.7902]
  novelty    10.4550  [10.4379, 10.4734]

coverage: 0.1766  (2080/11777 articles)


In [12]:
import glob
# any precomputed article-embedding artifacts?
print("word2vec:", glob.glob("/kaggle/input/**/*word2vec*", recursive=True))
print("bert dir:", glob.glob("/kaggle/input/**/*bert*", recursive=True))
# EB-NeRD ships embeddings as parquet with an 'article_id' + vector column
for p in glob.glob("/kaggle/input/**/*.parquet", recursive=True):
    if any(k in p.lower() for k in ("embed","word2vec","bert","contrast","vector")):
        print("candidate embedding parquet:", p)

word2vec: ['/kaggle/input/datasets/wrathofgod123/ebnerd/Ekstra_Bladet_word2vec', '/kaggle/input/datasets/wrathofgod123/ebnerd/Ekstra_Bladet_word2vec/Ekstra_Bladet_word2vec']
bert dir: ['/kaggle/input/datasets/wrathofgod123/ebnerd/google_bert_base_multilingual_cased', '/kaggle/input/datasets/wrathofgod123/ebnerd/google_bert_base_multilingual_cased/google_bert_base_multilingual_cased', '/kaggle/input/datasets/wrathofgod123/ebnerd/google_bert_base_multilingual_cased/google_bert_base_multilingual_cased/bert_base_multilingual_cased.parquet']
candidate embedding parquet: /kaggle/input/datasets/wrathofgod123/ebnerd/google_bert_base_multilingual_cased/google_bert_base_multilingual_cased/bert_base_multilingual_cased.parquet
candidate embedding parquet: /kaggle/input/datasets/wrathofgod123/ebnerd/Ekstra_Bladet_word2vec/Ekstra_Bladet_word2vec/document_vector.parquet


In [13]:
import polars as pl

BERT = "/kaggle/input/datasets/wrathofgod123/ebnerd/google_bert_base_multilingual_cased/google_bert_base_multilingual_cased/bert_base_multilingual_cased.parquet"
W2V  = "/kaggle/input/datasets/wrathofgod123/ebnerd/Ekstra_Bladet_word2vec/Ekstra_Bladet_word2vec/document_vector.parquet"

for name, path in [("BERT", BERT), ("word2vec", W2V)]:
    df = pl.read_parquet(path)
    print(f"\n=== {name} ===")
    print("rows:", df.height, "| cols:", df.columns)
    print(df.head(2))
    # dimensionality of the vector column
    for c in df.columns:
        v = df[c][0]
        if isinstance(v, (list,)) or (hasattr(v, "__len__") and not isinstance(v, str)):
            print(f"  {c}: vector dim = {len(v)}")


=== BERT ===
rows: 125541 | cols: ['article_id', 'google-bert/bert-base-multilingual-cased']
shape: (2, 2)
┌────────────┬─────────────────────────────────┐
│ article_id ┆ google-bert/bert-base-multilin… │
│ ---        ┆ ---                             │
│ i32        ┆ list[f32]                       │
╞════════════╪═════════════════════════════════╡
│ 3000022    ┆ [-0.350606, 0.003437, … 0.0019… │
│ 3000063    ┆ [-0.003448, 0.227659, … -0.057… │
└────────────┴─────────────────────────────────┘
  google-bert/bert-base-multilingual-cased: vector dim = 768

=== word2vec ===
rows: 125541 | cols: ['article_id', 'document_vector']
shape: (2, 2)
┌────────────┬─────────────────────────────────┐
│ article_id ┆ document_vector                 │
│ ---        ┆ ---                             │
│ i32        ┆ list[f32]                       │
╞════════════╪═════════════════════════════════╡
│ 3000022    ┆ [0.065424, -0.047425, … 0.0357… │
│ 3000063    ┆ [0.028815, -0.000166, … 0.0271… │
└────────

In [15]:
!pip install faiss-cpu -q

/usr/lib/python3.12/pty.py:95: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 64.9 MB/s eta 0:00:00:00:0100:01


In [16]:
import numpy as np, polars as pl, glob
import faiss

DEMO = glob.glob("/kaggle/input/**/ebnerd_demo", recursive=True)[0]
PREFIX="ebnerd"
def _prefix(c): return pl.concat_str([pl.lit(f"{PREFIX}:"), c.cast(pl.Utf8)])

W2V  = "/kaggle/input/datasets/wrathofgod123/ebnerd/Ekstra_Bladet_word2vec/Ekstra_Bladet_word2vec/document_vector.parquet"
BERT = "/kaggle/input/datasets/wrathofgod123/ebnerd/google_bert_base_multilingual_cased/google_bert_base_multilingual_cased/bert_base_multilingual_cased.parquet"

# ---- pick which embedding to run ----
EMB_PATH, EMB_COL = W2V, "document_vector"      # swap to BERT, "google-bert/..." for the other

# ---- parse impressions + history + article pool (demo) ----
a = pl.read_parquet(f"{DEMO}/articles.parquet")
demo_article_ids = set(a["article_id"].to_list())    # raw int ids in demo
b = pl.read_parquet(f"{DEMO}/train/behaviors.parquet")
imps = b.select(user_id=_prefix(pl.col("user_id")), timestamp="impression_time",
    candidate_ids=pl.col("article_ids_inview").list.eval(_prefix(pl.element())),
    labels=pl.col("article_ids_clicked").list.eval(_prefix(pl.element())))
h = pl.read_parquet(f"{DEMO}/train/history.parquet")
hist = h.select(user_id=_prefix(pl.col("user_id")),
    article_ids=pl.col("article_id_fixed").list.eval(_prefix(pl.element())),
    timestamps="impression_time_fixed")
pub_lut = {f"{PREFIX}:{aid}": pt for aid,pt in
           a.select(["article_id","published_time"]).iter_rows()}

# ---- load embeddings, filter to demo pool, build aligned matrix ----
edf = pl.read_parquet(EMB_PATH).filter(pl.col("article_id").is_in(demo_article_ids))
ids = [f"{PREFIX}:{i}" for i in edf["article_id"].to_list()]
emb = np.vstack(edf[EMB_COL].to_list()).astype("float32")
# L2-normalize so inner product == cosine
emb /= (np.linalg.norm(emb, axis=1, keepdims=True) + 1e-9)
ids_arr = np.array(ids); id_to_row = {a:i for i,a in enumerate(ids)}
emb_by_id = {aid: emb[i] for i,aid in enumerate(ids)}
pub_arr = np.array([pub_lut.get(a) for a in ids])
print(f"embedding: {EMB_COL} | articles in demo pool: {emb.shape}")

# ---- FAISS index (cosine via inner product) ----
index = faiss.IndexFlatIP(emb.shape[1]); index.add(emb)

# ---- user vector = mean-pooled history embeddings (leakage-guarded) ----
hist_lut = {u:(ai or [],ts or []) for u,ai,ts in
            hist.select(["user_id","article_ids","timestamps"]).iter_rows()}
def user_vec(uid, ts, max_hist=30):
    ai,tss = hist_lut.get(uid,([],[]))
    if not ai: return None
    if ts and tss and len(tss)==len(ai):
        kept=[(x,t) for x,t in zip(ai,tss) if t is not None and t<ts]
        kept.sort(key=lambda z:z[1]); ai=[x for x,_ in kept][-max_hist:]
    else: ai=ai[-max_hist:]
    vecs=[emb_by_id[x] for x in ai if x in emb_by_id]
    if not vecs: return None
    v=np.mean(vecs,axis=0); v/=(np.linalg.norm(v)+1e-9)
    return v.astype("float32")

# ---- recall@K via FAISS ----
Ks=(50,100,200); maxK=200
uvecs,u_labels,u_ts=[],[],[]
for imp in imps.iter_rows(named=True):
    labs=imp["labels"]
    if not labs: continue
    v=user_vec(imp["user_id"], imp["timestamp"])
    if v is None: continue
    uvecs.append(v); u_labels.append(set(labs)); u_ts.append(imp["timestamp"])
uvecs=np.vstack(uvecs)
print("queries to score:", len(uvecs))

D,I = index.search(uvecs, maxK*3)   # over-retrieve for freshness filtering
hits={k:0 for k in Ks}
for i in range(len(uvecs)):
    rows=I[i]; ts=u_ts[i]
    if ts is not None:
        rows=[r for r in rows if pub_arr[r] is None or pub_arr[r]<ts]
    ranked=ids_arr[rows[:maxK]]
    for k in Ks:
        if u_labels[i] & set(ranked[:k].tolist()): hits[k]+=1
n=len(uvecs)
print(f"\n=== SEMANTIC ({EMB_COL}) recall@K ===")
for k in Ks: print(f"recall@{k}: {hits[k]/n:.4f}")
print("=== BM25 baseline ===  recall@50: 0.0145 | @100: 0.0268 | @200: 0.0383")

embedding: document_vector | articles in demo pool: (11777, 300)
queries to score: 24724

=== SEMANTIC (document_vector) recall@K ===
recall@50: 0.0090
recall@100: 0.0180
recall@200: 0.0374
=== BM25 baseline ===  recall@50: 0.0145 | @100: 0.0268 | @200: 0.0383


In [17]:
!pip install faiss-cpu -q

import numpy as np, polars as pl, glob
import faiss

DEMO = glob.glob("/kaggle/input/**/ebnerd_demo", recursive=True)[0]
PREFIX="ebnerd"
def _prefix(c): return pl.concat_str([pl.lit(f"{PREFIX}:"), c.cast(pl.Utf8)])

BERT = "/kaggle/input/datasets/wrathofgod123/ebnerd/google_bert_base_multilingual_cased/google_bert_base_multilingual_cased/bert_base_multilingual_cased.parquet"

# ---- BERT embedding ----
EMB_PATH, EMB_COL = BERT, "google-bert/bert-base-multilingual-cased"

# ---- parse impressions + history + article pool (demo) ----
a = pl.read_parquet(f"{DEMO}/articles.parquet")
demo_article_ids = set(a["article_id"].to_list())
b = pl.read_parquet(f"{DEMO}/train/behaviors.parquet")
imps = b.select(user_id=_prefix(pl.col("user_id")), timestamp="impression_time",
    candidate_ids=pl.col("article_ids_inview").list.eval(_prefix(pl.element())),
    labels=pl.col("article_ids_clicked").list.eval(_prefix(pl.element())))
h = pl.read_parquet(f"{DEMO}/train/history.parquet")
hist = h.select(user_id=_prefix(pl.col("user_id")),
    article_ids=pl.col("article_id_fixed").list.eval(_prefix(pl.element())),
    timestamps="impression_time_fixed")
pub_lut = {f"{PREFIX}:{aid}": pt for aid,pt in
           a.select(["article_id","published_time"]).iter_rows()}

# ---- load embeddings, filter to demo pool, normalize ----
edf = pl.read_parquet(EMB_PATH).filter(pl.col("article_id").is_in(demo_article_ids))
ids = [f"{PREFIX}:{i}" for i in edf["article_id"].to_list()]
emb = np.vstack(edf[EMB_COL].to_list()).astype("float32")
emb /= (np.linalg.norm(emb, axis=1, keepdims=True) + 1e-9)
ids_arr = np.array(ids); id_to_row = {a:i for i,a in enumerate(ids)}
emb_by_id = {aid: emb[i] for i,aid in enumerate(ids)}
pub_arr = np.array([pub_lut.get(a) for a in ids])
print(f"embedding: {EMB_COL} | articles in demo pool: {emb.shape}")

# ---- FAISS index ----
index = faiss.IndexFlatIP(emb.shape[1]); index.add(emb)

# ---- user vector = mean-pooled history embeddings (leakage-guarded) ----
hist_lut = {u:(ai or [],ts or []) for u,ai,ts in
            hist.select(["user_id","article_ids","timestamps"]).iter_rows()}
def user_vec(uid, ts, max_hist=30):
    ai,tss = hist_lut.get(uid,([],[]))
    if not ai: return None
    if ts and tss and len(tss)==len(ai):
        kept=[(x,t) for x,t in zip(ai,tss) if t is not None and t<ts]
        kept.sort(key=lambda z:z[1]); ai=[x for x,_ in kept][-max_hist:]
    else: ai=ai[-max_hist:]
    vecs=[emb_by_id[x] for x in ai if x in emb_by_id]
    if not vecs: return None
    v=np.mean(vecs,axis=0); v/=(np.linalg.norm(v)+1e-9)
    return v.astype("float32")

# ---- recall@K via FAISS ----
Ks=(50,100,200); maxK=200
uvecs,u_labels,u_ts=[],[],[]
for imp in imps.iter_rows(named=True):
    labs=imp["labels"]
    if not labs: continue
    v=user_vec(imp["user_id"], imp["timestamp"])
    if v is None: continue
    uvecs.append(v); u_labels.append(set(labs)); u_ts.append(imp["timestamp"])
uvecs=np.vstack(uvecs)
print("queries to score:", len(uvecs))

D,I = index.search(uvecs, maxK*3)
hits={k:0 for k in Ks}
for i in range(len(uvecs)):
    rows=I[i]; ts=u_ts[i]
    if ts is not None:
        rows=[r for r in rows if pub_arr[r] is None or pub_arr[r]<ts]
    ranked=ids_arr[rows[:maxK]]
    for k in Ks:
        if u_labels[i] & set(ranked[:k].tolist()): hits[k]+=1
n=len(uvecs)
print(f"\n=== SEMANTIC ({EMB_COL}) recall@K ===")
for k in Ks: print(f"recall@{k}: {hits[k]/n:.4f}")
print("\n=== comparison ===")
print("BM25:     recall@50: 0.0145 | @100: 0.0268 | @200: 0.0383")
print("word2vec: recall@50: 0.0090 | @100: 0.0180 | @200: 0.0374")

embedding: google-bert/bert-base-multilingual-cased | articles in demo pool: (11777, 768)
queries to score: 24724

=== SEMANTIC (google-bert/bert-base-multilingual-cased) recall@K ===
recall@50: 0.0070
recall@100: 0.0139
recall@200: 0.0301

=== comparison ===
BM25:     recall@50: 0.0145 | @100: 0.0268 | @200: 0.0383
word2vec: recall@50: 0.0090 | @100: 0.0180 | @200: 0.0374


In [18]:
import glob
for p in glob.glob("/kaggle/input/**/*.parquet", recursive=True):
    if any(k in p.lower() for k in ("contrast","emb","vector","roberta","xlm")):
        print(p)

/kaggle/input/datasets/wrathofgod123/ebnerd/Ekstra_Bladet_word2vec/Ekstra_Bladet_word2vec/document_vector.parquet


In [1]:
!pip install sentence-transformers faiss-cpu -q

import numpy as np, polars as pl, glob, torch
from sentence_transformers import SentenceTransformer
import faiss

print("GPU:", torch.cuda.is_available())          # must be True
DEMO = glob.glob("/kaggle/input/**/ebnerd_demo", recursive=True)[0]
PREFIX="ebnerd"
def _prefix(c): return pl.concat_str([pl.lit(f"{PREFIX}:"), c.cast(pl.Utf8)])

# ---- parse ----
a = pl.read_parquet(f"{DEMO}/articles.parquet")
articles = a.select(article_id=_prefix(pl.col("article_id")),
    title=pl.col("title").fill_null(""), abstract=pl.col("subtitle").fill_null(""),
    published_time="published_time")
b = pl.read_parquet(f"{DEMO}/train/behaviors.parquet")
imps = b.select(user_id=_prefix(pl.col("user_id")), timestamp="impression_time",
    candidate_ids=pl.col("article_ids_inview").list.eval(_prefix(pl.element())),
    labels=pl.col("article_ids_clicked").list.eval(_prefix(pl.element())))
h = pl.read_parquet(f"{DEMO}/train/history.parquet")
hist = h.select(user_id=_prefix(pl.col("user_id")),
    article_ids=pl.col("article_id_fixed").list.eval(_prefix(pl.element())),
    timestamps="impression_time_fixed")
pub_lut = {f"{PREFIX}:{aid}": pt for aid,pt in
           a.select(["article_id","published_time"]).iter_rows()}

# ---- encode articles with E5 (passage: prefix, even for Danish) ----
model = SentenceTransformer("intfloat/multilingual-e5-base", device="cuda")
ids, texts = [], []
for aid, ti, ab in articles.select(["article_id","title","abstract"]).iter_rows():
    ids.append(aid); texts.append(f"passage: {ti} {ab}".strip())
emb = model.encode(texts, batch_size=256, normalize_embeddings=True,
                   show_progress_bar=True, convert_to_numpy=True).astype("float32")
ids_arr = np.array(ids); id_to_row = {a:i for i,a in enumerate(ids)}
emb_by_id = {aid: emb[i] for i,aid in enumerate(ids)}
pub_arr = np.array([pub_lut.get(a) for a in ids])
print("article embeddings:", emb.shape)

index = faiss.IndexFlatIP(emb.shape[1]); index.add(emb)

# ---- user vector: mean-pool history article embeddings (leakage-guarded) ----
# NOTE: history articles are encoded as passages; the pooled user vector is the
# "query". E5's query: prefix applies to text, but since we pool precomputed
# passage embeddings, we use them directly (asymmetric prefix approximated).
hist_lut = {u:(ai or [],ts or []) for u,ai,ts in
            hist.select(["user_id","article_ids","timestamps"]).iter_rows()}
def user_vec(uid, ts, max_hist=30):
    ai,tss = hist_lut.get(uid,([],[]))
    if not ai: return None
    if ts and tss and len(tss)==len(ai):
        kept=[(x,t) for x,t in zip(ai,tss) if t is not None and t<ts]
        kept.sort(key=lambda z:z[1]); ai=[x for x,_ in kept][-max_hist:]
    else: ai=ai[-max_hist:]
    vecs=[emb_by_id[x] for x in ai if x in emb_by_id]
    if not vecs: return None
    v=np.mean(vecs,axis=0); v/=(np.linalg.norm(v)+1e-9)
    return v.astype("float32")

# ---- recall@K ----
Ks=(50,100,200); maxK=200
uvecs,u_labels,u_ts=[],[],[]
for imp in imps.iter_rows(named=True):
    labs=imp["labels"]
    if not labs: continue
    v=user_vec(imp["user_id"], imp["timestamp"])
    if v is None: continue
    uvecs.append(v); u_labels.append(set(labs)); u_ts.append(imp["timestamp"])
uvecs=np.vstack(uvecs)
print("queries to score:", len(uvecs))

D,I = index.search(uvecs, maxK*3)
hits={k:0 for k in Ks}
for i in range(len(uvecs)):
    rows=I[i]; ts=u_ts[i]
    if ts is not None:
        rows=[r for r in rows if pub_arr[r] is None or pub_arr[r]<ts]
    ranked=ids_arr[rows[:maxK]]
    for k in Ks:
        if u_labels[i] & set(ranked[:k].tolist()): hits[k]+=1
n=len(uvecs)
print(f"\n=== SEMANTIC (E5) recall@K ===")
for k in Ks: print(f"recall@{k}: {hits[k]/n:.4f}")
print("\n=== full comparison ===")
print("BM25:     @50: 0.0145 | @100: 0.0268 | @200: 0.0383")
print("word2vec: @50: 0.0090 | @100: 0.0180 | @200: 0.0374")
print("mBERT:    @50: 0.0070 | @100: 0.0139 | @200: 0.0301")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 87.8 MB/s eta 0:00:00:00:0100:01
GPU: True


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Batches:   0%|          | 0/47 [00:00<?, ?it/s]

article embeddings: (11777, 768)
queries to score: 24724

=== SEMANTIC (E5) recall@K ===
recall@50: 0.0101
recall@100: 0.0190
recall@200: 0.0368

=== full comparison ===
BM25:     @50: 0.0145 | @100: 0.0268 | @200: 0.0383
word2vec: @50: 0.0090 | @100: 0.0180 | @200: 0.0374
mBERT:    @50: 0.0070 | @100: 0.0139 | @200: 0.0301


In [2]:
!pip install sentence-transformers faiss-cpu bm25s -q
import re, math, numpy as np, polars as pl, glob, torch
from collections import Counter
from sentence_transformers import SentenceTransformer

DEMO = glob.glob("/kaggle/input/**/ebnerd_demo", recursive=True)[0]
W2V  = "/kaggle/input/datasets/wrathofgod123/ebnerd/Ekstra_Bladet_word2vec/Ekstra_Bladet_word2vec/document_vector.parquet"
BERT = "/kaggle/input/datasets/wrathofgod123/ebnerd/google_bert_base_multilingual_cased/google_bert_base_multilingual_cased/bert_base_multilingual_cased.parquet"
PREFIX="ebnerd"
def _prefix(c): return pl.concat_str([pl.lit(f"{PREFIX}:"), c.cast(pl.Utf8)])
_WORD=re.compile(r"[^\W\d_]+",re.UNICODE)
def tok(t): return _WORD.findall(t.lower()) if t else []

# ---------- parse ----------
a=pl.read_parquet(f"{DEMO}/articles.parquet")
demo_ids=set(a["article_id"].to_list())
articles=a.select(article_id=_prefix(pl.col("article_id")),
    title=pl.col("title").fill_null(""), abstract=pl.col("subtitle").fill_null(""),
    category_str=pl.col("category_str").fill_null(""))
b=pl.read_parquet(f"{DEMO}/train/behaviors.parquet")
imps_full=b.select(user_id=_prefix(pl.col("user_id")), timestamp="impression_time",
    candidate_ids=pl.col("article_ids_inview").list.eval(_prefix(pl.element())),
    labels=pl.col("article_ids_clicked").list.eval(_prefix(pl.element())))
h=pl.read_parquet(f"{DEMO}/train/history.parquet")
hist=h.select(user_id=_prefix(pl.col("user_id")),
    article_ids=pl.col("article_id_fixed").list.eval(_prefix(pl.element())),
    timestamps="impression_time_fixed")

cat_lut={aid:c for aid,c in articles.select(["article_id","category_str"]).iter_rows()}
hist_lut={u:(ai or [],ts or []) for u,ai,ts in hist.select(["user_id","article_ids","timestamps"]).iter_rows()}
pop=Counter()
for (labs,) in imps_full.select("labels").iter_rows():
    for x in (labs or []): pop[x]+=1
total_pop=sum(pop.values())
sbp=sorted(pop.items(),key=lambda kv:-kv[1]); n_head=max(1,int(0.2*len(sbp)))
head_set={aid for aid,_ in sbp[:n_head]}

# ---------- BM25 setup ----------
ids=[]; corpus=[]
for aid,ti,ab in articles.select(["article_id","title","abstract"]).iter_rows():
    ids.append(aid); corpus.append(tok(f"{ti} {ab}"))
id_to_row={x:i for i,x in enumerate(ids)}
title_lut={aid:tok(ti) for aid,ti in articles.select(["article_id","title"]).iter_rows()}
class BM25:
    def __init__(s,c,k1=1.5,b=0.75):
        s.k1,s.b=k1,b; s.N=len(c); s.tf=[Counter(d) for d in c]
        s.dl=np.array([len(d) for d in c],float); s.avg=s.dl.mean()
        df=Counter()
        for t in s.tf: df.update(t.keys())
        s.idf={w:math.log((s.N-d+.5)/(d+.5)+1) for w,d in df.items()}
    def score(s,q,r):
        if not q: return 0.0
        tf=s.tf[r]; dn=s.k1*(1-s.b+s.b*s.dl[r]/s.avg); v=0.0
        for w in set(q):
            f=tf.get(w,0)
            if f: v+=s.idf.get(w,0)*(f*(s.k1+1))/(f+dn)
        return v
bm25=BM25(corpus)
def hist_q(uid,ts,mh=30):
    ai,tss=hist_lut.get(uid,([],[]))
    if not ai: return []
    if ts and tss and len(tss)==len(ai):
        k=[(x,t) for x,t in zip(ai,tss) if t is not None and t<ts]; k.sort(key=lambda z:z[1]); ai=[x for x,_ in k][-mh:]
    else: ai=ai[-mh:]
    q=[]; [q.extend(title_lut.get(x,[])) for x in ai]; return q

# ---------- dense embeddings (word2vec, mBERT, E5) ----------
def load_parquet_emb(path,col):
    e=pl.read_parquet(path).filter(pl.col("article_id").is_in(demo_ids))
    d={f"{PREFIX}:{i}":np.array(v,dtype="float32") for i,v in zip(e["article_id"].to_list(),e[col].to_list())}
    for k in d: d[k]/=(np.linalg.norm(d[k])+1e-9)
    return d
emb_w2v =load_parquet_emb(W2V,"document_vector")
emb_bert=load_parquet_emb(BERT,"google-bert/bert-base-multilingual-cased")
# E5 (re-encode)
model=SentenceTransformer("intfloat/multilingual-e5-base",device="cuda")
e5_ids=[aid for aid,_ in articles.select(["article_id","title"]).iter_rows()]
e5_txt=[f"passage: {ti} {ab}".strip() for _,ti,ab in articles.select(["article_id","title","abstract"]).iter_rows()]
e5_mat=model.encode(e5_txt,batch_size=256,normalize_embeddings=True,convert_to_numpy=True,show_progress_bar=True)
emb_e5={aid:e5_mat[i] for i,aid in enumerate(e5_ids)}

def make_user_vec(emb_by_id):
    def uv(uid,ts,mh=30):
        ai,tss=hist_lut.get(uid,([],[]))
        if not ai: return None
        if ts and tss and len(tss)==len(ai):
            k=[(x,t) for x,t in zip(ai,tss) if t is not None and t<ts]; k.sort(key=lambda z:z[1]); ai=[x for x,_ in k][-mh:]
        else: ai=ai[-mh:]
        vs=[emb_by_id[x] for x in ai if x in emb_by_id]
        if not vs: return None
        v=np.mean(vs,0); v/=(np.linalg.norm(v)+1e-9); return v
    return uv

# ---------- scorers: (candidate_ids, imp) -> scores ----------
def bm25_scorer(cids,imp):
    q=hist_q(imp["user_id"],imp["timestamp"])
    return np.array([bm25.score(q,id_to_row[c]) if c in id_to_row else 0.0 for c in cids])
def dense_scorer(emb_by_id):
    uv=make_user_vec(emb_by_id)
    def f(cids,imp):
        v=uv(imp["user_id"],imp["timestamp"])
        if v is None: return np.zeros(len(cids))
        return np.array([float(v@emb_by_id[c]) if c in emb_by_id else 0.0 for c in cids])
    return f
SCORERS={"BM25":bm25_scorer,"word2vec":dense_scorer(emb_w2v),
         "mBERT":dense_scorer(emb_bert),"E5":dense_scorer(emb_e5)}

# ---------- metrics ----------
def auc_i(sc,lb):
    p=lb==1;n=lb==0;np_,nn=p.sum(),n.sum()
    if np_==0 or nn==0: return None
    o=np.argsort(sc);r=np.empty_like(o,float);r[o]=np.arange(1,len(sc)+1)
    return float((r[p].sum()-np_*(np_+1)/2)/(np_*nn))
def mrr_i(sc,lb):
    rl=lb[np.argsort(-sc)];idx=np.nonzero(rl==1)[0];return 1/(idx[0]+1) if len(idx) else 0.0
def dcg(r): return np.sum(r/np.log2(np.arange(2,len(r)+2)))
def ndcg_i(sc,lb,k):
    rk=lb[np.argsort(-sc)][:k];il=np.sort(lb)[::-1][:k];i=dcg(il);return float(dcg(rk)/i) if i>0 else 0.0

def run(scorer,rec_k=5,Ks=(5,10)):
    rows=[];recc=set()
    for imp in imps_full.iter_rows(named=True):
        labs=set(imp["labels"] or []);cand=imp["candidate_ids"]
        if not labs or not cand: continue
        y=np.array([1 if c in labs else 0 for c in cand])
        if y.sum()==0: continue
        sc=np.asarray(scorer(cand,imp),float)
        d={"auc":auc_i(sc,y),"mrr":mrr_i(sc,y)}
        for k in Ks: d[f"ndcg@{k}"]=ndcg_i(sc,y,k)
        top=np.array(cand)[np.argsort(-sc)[:rec_k]];recc.update(top.tolist())
        cats=[cat_lut.get(x) for x in top]
        if len(cats)>1:
            same=sum(1 for i in range(len(cats)) for j in range(i+1,len(cats)) if cats[i] and cats[i]==cats[j])
            d["diversity"]=1-same/(len(cats)*(len(cats)-1)/2)
        nv=[-math.log2(pop.get(x,1)/total_pop) for x in top if total_pop]
        d["novelty"]=float(np.mean(nv)) if nv else 0.0
        d["_slice"]="head" if (labs&head_set) else "tail"
        rows.append(d)
    return rows,recc

mkeys=["auc","mrr","ndcg@5","ndcg@10"]
def agg(sub,nb=500,seed=0):
    rng=np.random.default_rng(seed);o={}
    for m in mkeys:
        v=np.array([r[m] for r in sub if r.get(m) is not None],float)
        bt=np.array([rng.choice(v,len(v),replace=True).mean() for _ in range(nb)])
        o[m]=(v.mean(),*np.percentile(bt,[2.5,97.5]))
    return o

# ---------- run all four ----------
print(f"{'method':9s} {'slice':5s} {'AUC':>18s} {'MRR':>10s} {'nDCG@5':>10s} {'nDCG@10':>10s}  cov")
for name,scorer in SCORERS.items():
    rows,recc=run(scorer)
    cov=len(recc)/len(ids)
    for sl,sub in [("all",rows),("head",[r for r in rows if r['_slice']=='head']),
                   ("tail",[r for r in rows if r['_slice']=='tail'])]:
        A=agg(sub)
        auc=f"{A['auc'][0]:.3f}[{A['auc'][1]:.3f},{A['auc'][2]:.3f}]"
        print(f"{name:9s} {sl:5s} {auc:>18s} {A['mrr'][0]:>10.3f} {A['ndcg@5'][0]:>10.3f} {A['ndcg@10'][0]:>10.3f}  {cov:.3f}")
    print()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.7/74.7 kB 4.7 MB/s eta 0:00:00


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/47 [00:00<?, ?it/s]

method    slice                AUC        MRR     nDCG@5    nDCG@10  cov
BM25      all   0.519[0.515,0.523]      0.339      0.373      0.456  0.177
BM25      head  0.531[0.525,0.537]      0.349      0.385      0.466  0.177
BM25      tail  0.508[0.502,0.513]      0.330      0.363      0.447  0.177

word2vec  all   0.510[0.506,0.514]      0.334      0.365      0.451  0.178
word2vec  head  0.521[0.516,0.527]      0.346      0.379      0.462  0.178
word2vec  tail  0.499[0.493,0.504]      0.323      0.353      0.440  0.178

mBERT     all   0.487[0.483,0.491]      0.322      0.351      0.439  0.174
mBERT     head  0.515[0.509,0.521]      0.346      0.379      0.462  0.174
mBERT     tail  0.462[0.456,0.467]      0.299      0.324      0.417  0.174

E5        all   0.517[0.513,0.521]      0.334      0.371      0.453  0.172
E5        head  0.535[0.529,0.540]      0.345      0.385      0.466  0.172
E5        tail  0.500[0.494,0.506]      0.322      0.357      0.440  0.172



In [1]:
import datetime as dt
from bisect import bisect_left
from collections import defaultdict
import numpy as np

# ---- recency ----
def recency_score(pub, imp_t, tau_hours=6.0):
    if pub is None or imp_t is None: return 0.0
    dh = (imp_t - pub).total_seconds()/3600.0
    return float(np.exp(-dh/tau_hours)) if dh>=0 else 0.0

# ---- leakage-safe popularity ----
class PopularityIndex:
    def __init__(self, click_events):
        self.by = defaultdict(list)
        for aid,t in click_events:
            if t is not None: self.by[aid].append(t)
        for aid in self.by: self.by[aid].sort()
    def count_before(self, aid, T, window_hours=24.0):
        times=self.by.get(aid)
        if not times: return 0
        lo=T-dt.timedelta(hours=window_hours)
        return bisect_left(times,T)-bisect_left(times,lo)   # [T-window, T)
    def total_before(self, aid, T):
        times=self.by.get(aid)
        return bisect_left(times,T) if times else 0

# ---- build click events from TRAIN impressions only (leakage: never use val/test clicks) ----
click_events=[]
for imp in imps_full.iter_rows(named=True):     # imps_full = demo train impressions
    T=imp["timestamp"]
    for aid in (imp["labels"] or []):
        click_events.append((aid, T))
popidx = PopularityIndex(click_events)
print("articles with any clicks:", len(popidx.by))
print("total click events:", len(click_events))

# ---- verification: popularity at T must exclude the impression's own click ----
# pick an impression, confirm its clicked article's pop-before < pop-total-including-it
sample = next(imps_full.iter_rows(named=True))
T=sample["timestamp"]; clicked=(sample["labels"] or [None])[0]
if clicked:
    before = popidx.total_before(clicked, T)
    at_or_after = sum(1 for a,t in click_events if a==clicked and t>=T)
    print(f"\nclicked={clicked} @ {T}")
    print(f"  clicks strictly before T: {before}")
    print(f"  clicks at/after T (excluded): {at_or_after}  <- includes this very click, correctly excluded")

# ---- distribution sanity ----
import statistics
pops=[popidx.count_before(a, T, 24) for a in list(popidx.by)[:2000]]
print(f"\npopularity(24h) over sample: mean={statistics.mean(pops):.2f}, max={max(pops)}")

NameError: name 'imps_full' is not defined

In [2]:
import datetime as dt, numpy as np, polars as pl, glob
from bisect import bisect_left
from collections import defaultdict
import statistics

DEMO = glob.glob("/kaggle/input/**/ebnerd_demo", recursive=True)[0]
PREFIX="ebnerd"
def _prefix(c): return pl.concat_str([pl.lit(f"{PREFIX}:"), c.cast(pl.Utf8)])

# ---- parse demo train impressions ----
b = pl.read_parquet(f"{DEMO}/train/behaviors.parquet")
imps_full = b.select(
    impression_id="impression_id", user_id=_prefix(pl.col("user_id")),
    timestamp="impression_time",
    candidate_ids=pl.col("article_ids_inview").list.eval(_prefix(pl.element())),
    labels=pl.col("article_ids_clicked").list.eval(_prefix(pl.element())),
)

# ---- recency ----
def recency_score(pub, imp_t, tau_hours=6.0):
    if pub is None or imp_t is None: return 0.0
    dh = (imp_t - pub).total_seconds()/3600.0
    return float(np.exp(-dh/tau_hours)) if dh>=0 else 0.0

# ---- leakage-safe popularity ----
class PopularityIndex:
    def __init__(self, click_events):
        self.by = defaultdict(list)
        for aid,t in click_events:
            if t is not None: self.by[aid].append(t)
        for aid in self.by: self.by[aid].sort()
    def count_before(self, aid, T, window_hours=24.0):
        times=self.by.get(aid)
        if not times: return 0
        lo=T-dt.timedelta(hours=window_hours)
        return bisect_left(times,T)-bisect_left(times,lo)
    def total_before(self, aid, T):
        times=self.by.get(aid)
        return bisect_left(times,T) if times else 0

# ---- build click events from TRAIN impressions only ----
click_events=[]
for imp in imps_full.iter_rows(named=True):
    T=imp["timestamp"]
    for aid in (imp["labels"] or []):
        click_events.append((aid, T))
popidx = PopularityIndex(click_events)
print("articles with any clicks:", len(popidx.by))
print("total click events:", len(click_events))

# ---- verification: popularity at T excludes the impression's own click ----
sample = next(imps_full.iter_rows(named=True))
T=sample["timestamp"]; clicked=(sample["labels"] or [None])[0]
if clicked:
    before = popidx.total_before(clicked, T)
    at_or_after = sum(1 for a,t in click_events if a==clicked and t>=T)
    print(f"\nclicked={clicked} @ {T}")
    print(f"  clicks strictly before T: {before}")
    print(f"  clicks at/after T (excluded): {at_or_after}  <- includes this very click, correctly excluded")

# ---- distribution sanity ----
pops=[popidx.count_before(a, T, 24) for a in list(popidx.by)[:2000]]
print(f"\npopularity(24h) over sample: mean={statistics.mean(pops):.2f}, max={max(pops)}")

articles with any clicks: 1114
total click events: 24888

clicked=ebnerd:9759966 @ 2023-05-21 21:06:50
  clicks strictly before T: 21
  clicks at/after T (excluded): 19  <- includes this very click, correctly excluded

popularity(24h) over sample: mean=3.21, max=119


In [3]:
# ---- article pool + published times (for recency retriever) ----
a = pl.read_parquet(f"{DEMO}/articles.parquet")
pool_ids = [f"{PREFIX}:{i}" for i in a["article_id"].to_list()]
pub_lut = {f"{PREFIX}:{aid}": pt for aid,pt in a.select(["article_id","published_time"]).iter_rows()}

# ---- history (for the leakage filter reused from before) ----
h = pl.read_parquet(f"{DEMO}/train/history.parquet")
hist = h.select(user_id=_prefix(pl.col("user_id")),
    article_ids=pl.col("article_id_fixed").list.eval(_prefix(pl.element())),
    timestamps="impression_time_fixed")
hist_lut = {u:(ai or [],ts or []) for u,ai,ts in
            hist.select(["user_id","article_ids","timestamps"]).iter_rows()}

Ks=(50,100,200); maxK=200

def eval_retriever(score_fn, name):
    """score_fn(T) -> np.array of scores aligned to pool_ids. Evaluated by recall@K."""
    hits={k:0 for k in Ks}; n=0
    pool_arr=np.array(pool_ids)
    for imp in imps_full.iter_rows(named=True):
        labs=imp["labels"]
        if not labs: continue
        # require history so it's comparable to content methods (same scored set)
        ai,ts=hist_lut.get(imp["user_id"],([],[]))
        if not ai: continue
        T=imp["timestamp"]
        scores=score_fn(T)
        # freshness: drop future-published
        for i,aid in enumerate(pool_ids):
            pt=pub_lut.get(aid)
            if pt is not None and pt>=T: scores[i]=-np.inf
        top=np.argpartition(-scores, maxK-1)[:maxK]
        top=top[np.argsort(-scores[top])]
        ranked=set(pool_arr[top][:maxK].tolist())
        n+=1
        lab=set(labs)
        for k in Ks:
            if lab & set(pool_arr[top][:k].tolist()): hits[k]+=1
    print(f"=== {name} ===")
    for k in Ks: print(f"recall@{k}: {hits[k]/n:.4f}")
    print(f"(scored {n} impressions)\n")

# ---- popularity-only retriever: rank all articles by clicks before T ----
def pop_scores(T):
    return np.array([popidx.total_before(aid, T) for aid in pool_ids], dtype=float)

# ---- recency-only retriever: rank all articles by freshness at T ----
def rec_scores(T):
    return np.array([recency_score(pub_lut.get(aid), T) for aid in pool_ids], dtype=float)

eval_retriever(pop_scores, "POPULARITY-only")
eval_retriever(rec_scores, "RECENCY-only")
print("=== content baselines (from Q2/Q3) ===")
print("BM25:     @50 0.0145 | @100 0.0268 | @200 0.0383")
print("E5:       @50 0.0101 | @100 0.0190 | @200 0.0368")

=== POPULARITY-only ===
recall@50: 0.1572
recall@100: 0.2644
recall@200: 0.4363
(scored 24724 impressions)

=== RECENCY-only ===
recall@50: 0.9142
recall@100: 0.9436
recall@200: 0.9597
(scored 24724 impressions)

=== content baselines (from Q2/Q3) ===
BM25:     @50 0.0145 | @100 0.0268 | @200 0.0383
E5:       @50 0.0101 | @100 0.0190 | @200 0.0368


In [4]:
# how many articles are published before a typical impression T?
sample_T = next(imps_full.iter_rows(named=True))["timestamp"]
n_eligible = sum(1 for aid in pool_ids
                 if pub_lut.get(aid) is not None and pub_lut.get(aid) < sample_T)
print(f"articles published before T: {n_eligible} / {len(pool_ids)}")

# distribution: how old is the clicked article, typically?
import numpy as np
ages=[]
for imp in imps_full.iter_rows(named=True):
    labs=imp["labels"]; T=imp["timestamp"]
    if not labs: continue
    pt=pub_lut.get(labs[0])
    if pt and T: ages.append((T-pt).total_seconds()/3600.0)
ages=np.array(ages)
print(f"clicked-article age (hours): median={np.median(ages):.1f}, "
      f"90th pct={np.percentile(ages,90):.1f}")
print(f"clicked within 24h of publish: {(ages<24).mean():.1%}")

articles published before T: 9668 / 11777
clicked-article age (hours): median=2.8, 90th pct=11.5
clicked within 24h of publish: 94.8%


In [5]:
import numpy as np

Ks=(50,100,200); maxK=200
pool_arr = np.array(pool_ids)
N = len(pool_ids)

# ---- precompute per-article recency & popularity are T-dependent, so per impression ----
# content needs the history query; precompute BM25 doc structures already in `bm25`

def minmax(x):
    lo, hi = x.min(), x.max()
    if hi - lo < 1e-12: return np.zeros_like(x)
    return (x - lo) / (hi - lo)

def fusion_eval(w_c, w_r, w_p, name, sample_n=4000):
    """Weighted fusion of content+recency+popularity. Sampled for speed."""
    hits={k:0 for k in Ks}; n=0
    # sample impressions for speed (full-pool scoring is heavy)
    rows = imps_full.sample(n=min(sample_n, imps_full.height), seed=0)
    # precompute published_time array aligned to pool
    pub_arr = np.array([pub_lut.get(a) for a in pool_ids])
    for imp in rows.iter_rows(named=True):
        labs=imp["labels"]
        if not labs: continue
        ai,ts = hist_lut.get(imp["user_id"],([],[]))
        if not ai: continue
        T=imp["timestamp"]

        # --- content: BM25 of history query vs every article ---
        q = hist_q(imp["user_id"], T)
        if w_c > 0 and q:
            c = np.array([bm25.score(q, id_to_row[a]) if a in id_to_row else 0.0
                          for a in pool_ids])
        else:
            c = np.zeros(N)

        # --- recency ---
        r = np.array([recency_score(pub_lut.get(a), T) for a in pool_ids]) if w_r>0 else np.zeros(N)

        # --- popularity (leakage-safe, clicks before T) ---
        p = np.array([popidx.total_before(a, T) for a in pool_ids], dtype=float) if w_p>0 else np.zeros(N)

        # --- normalize each within-impression, then weight ---
        score = w_c*minmax(c) + w_r*minmax(r) + w_p*minmax(p)

        # freshness filter
        for i,pt in enumerate(pub_arr):
            if pt is not None and pt >= T: score[i] = -np.inf

        top = np.argpartition(-score, maxK-1)[:maxK]
        top = top[np.argsort(-score[top])]
        ranked = pool_arr[top]
        lab=set(labs); n+=1
        for k in Ks:
            if lab & set(ranked[:k].tolist()): hits[k]+=1
    print(f"=== {name}  (w_c={w_c}, w_r={w_r}, w_p={w_p}, n={n}) ===")
    for k in Ks: print(f"recall@{k}: {hits[k]/n:.4f}")
    print()

# ---- run a few weightings ----
fusion_eval(0.0, 1.0, 0.0, "recency-only (check)")          # sanity vs 0.96
fusion_eval(0.0, 0.0, 1.0, "popularity-only (check)")       # sanity vs 0.44
fusion_eval(0.0, 0.7, 0.3, "recency+popularity")
fusion_eval(0.2, 0.6, 0.2, "content+recency+popularity")
fusion_eval(0.5, 0.3, 0.2, "content-heavy fusion")

NameError: name 'hist_q' is not defined

In [6]:
!pip install bm25s -q
import re, math, numpy as np, polars as pl, glob, datetime as dt
from bisect import bisect_left
from collections import defaultdict, Counter

DEMO = glob.glob("/kaggle/input/**/ebnerd_demo", recursive=True)[0]
PREFIX="ebnerd"
def _prefix(c): return pl.concat_str([pl.lit(f"{PREFIX}:"), c.cast(pl.Utf8)])
_WORD=re.compile(r"[^\W\d_]+",re.UNICODE)
def tok(t): return _WORD.findall(t.lower()) if t else []

# ---- parse ----
a=pl.read_parquet(f"{DEMO}/articles.parquet")
articles=a.select(article_id=_prefix(pl.col("article_id")),
    title=pl.col("title").fill_null(""), abstract=pl.col("subtitle").fill_null(""),
    published_time="published_time")
b=pl.read_parquet(f"{DEMO}/train/behaviors.parquet")
imps_full=b.select(user_id=_prefix(pl.col("user_id")), timestamp="impression_time",
    candidate_ids=pl.col("article_ids_inview").list.eval(_prefix(pl.element())),
    labels=pl.col("article_ids_clicked").list.eval(_prefix(pl.element())))
h=pl.read_parquet(f"{DEMO}/train/history.parquet")
hist=h.select(user_id=_prefix(pl.col("user_id")),
    article_ids=pl.col("article_id_fixed").list.eval(_prefix(pl.element())),
    timestamps="impression_time_fixed")

pool_ids=[f"{PREFIX}:{i}" for i in a["article_id"].to_list()]
pub_lut={f"{PREFIX}:{aid}":pt for aid,pt in a.select(["article_id","published_time"]).iter_rows()}
hist_lut={u:(ai or [],ts or []) for u,ai,ts in hist.select(["user_id","article_ids","timestamps"]).iter_rows()}

# ---- BM25 ----
ids=[]; corpus=[]
for aid,ti,ab in articles.select(["article_id","title","abstract"]).iter_rows():
    ids.append(aid); corpus.append(tok(f"{ti} {ab}"))
id_to_row={x:i for i,x in enumerate(ids)}
title_lut={aid:tok(ti) for aid,ti in articles.select(["article_id","title"]).iter_rows()}
class BM25:
    def __init__(s,c,k1=1.5,b=0.75):
        s.k1,s.b=k1,b; s.N=len(c); s.tf=[Counter(d) for d in c]
        s.dl=np.array([len(d) for d in c],float); s.avg=s.dl.mean()
        df=Counter()
        for t in s.tf: df.update(t.keys())
        s.idf={w:math.log((s.N-d+.5)/(d+.5)+1) for w,d in df.items()}
    def score(s,q,r):
        if not q: return 0.0
        tf=s.tf[r]; dn=s.k1*(1-s.b+s.b*s.dl[r]/s.avg); v=0.0
        for w in set(q):
            f=tf.get(w,0)
            if f: v+=s.idf.get(w,0)*(f*(s.k1+1))/(f+dn)
        return v
bm25=BM25(corpus)
def hist_q(uid,ts,mh=30):
    ai,tss=hist_lut.get(uid,([],[]))
    if not ai: return []
    if ts and tss and len(tss)==len(ai):
        k=[(x,t) for x,t in zip(ai,tss) if t is not None and t<ts]; k.sort(key=lambda z:z[1]); ai=[x for x,_ in k][-mh:]
    else: ai=ai[-mh:]
    q=[]; [q.extend(title_lut.get(x,[])) for x in ai]; return q

# ---- recency + popularity (leakage-safe) ----
def recency_score(pub,imp_t,tau=6.0):
    if pub is None or imp_t is None: return 0.0
    dh=(imp_t-pub).total_seconds()/3600.0
    return float(np.exp(-dh/tau)) if dh>=0 else 0.0
class PopIdx:
    def __init__(s,ev):
        s.by=defaultdict(list)
        for aid,t in ev:
            if t is not None: s.by[aid].append(t)
        for aid in s.by: s.by[aid].sort()
    def total_before(s,aid,T):
        t=s.by.get(aid); return bisect_left(t,T) if t else 0
click_events=[]
for imp in imps_full.iter_rows(named=True):
    for aid in (imp["labels"] or []): click_events.append((aid,imp["timestamp"]))
popidx=PopIdx(click_events)

print("setup complete — bm25, recency, popularity ready")

# ================= FUSION =================
Ks=(50,100,200); maxK=200
pool_arr=np.array(pool_ids); N=len(pool_ids)
pub_arr=np.array([pub_lut.get(a) for a in pool_ids])
def minmax(x):
    lo,hi=x.min(),x.max()
    return np.zeros_like(x) if hi-lo<1e-12 else (x-lo)/(hi-lo)

def fusion_eval(w_c,w_r,w_p,name,sample_n=4000):
    hits={k:0 for k in Ks}; n=0
    rows=imps_full.sample(n=min(sample_n,imps_full.height),seed=0)
    for imp in rows.iter_rows(named=True):
        labs=imp["labels"]
        if not labs: continue
        ai,ts=hist_lut.get(imp["user_id"],([],[]))
        if not ai: continue
        T=imp["timestamp"]
        c=np.array([bm25.score(hist_q(imp["user_id"],T),id_to_row[x]) if x in id_to_row else 0.0 for x in pool_ids]) if w_c>0 else np.zeros(N)
        r=np.array([recency_score(pub_lut.get(x),T) for x in pool_ids]) if w_r>0 else np.zeros(N)
        p=np.array([popidx.total_before(x,T) for x in pool_ids],float) if w_p>0 else np.zeros(N)
        score=w_c*minmax(c)+w_r*minmax(r)+w_p*minmax(p)
        for i,pt in enumerate(pub_arr):
            if pt is not None and pt>=T: score[i]=-np.inf
        top=np.argpartition(-score,maxK-1)[:maxK]; top=top[np.argsort(-score[top])]
        ranked=pool_arr[top]; lab=set(labs); n+=1
        for k in Ks:
            if lab & set(ranked[:k].tolist()): hits[k]+=1
    print(f"=== {name} (w_c={w_c},w_r={w_r},w_p={w_p}, n={n}) ===")
    for k in Ks: print(f"recall@{k}: {hits[k]/n:.4f}")
    print()

fusion_eval(0,1,0,"recency-only (check ~0.96)")
fusion_eval(0,0,1,"popularity-only (check ~0.44)")
fusion_eval(0,0.7,0.3,"recency+popularity")
fusion_eval(0.2,0.6,0.2,"content+recency+popularity")
fusion_eval(0.5,0.3,0.2,"content-heavy")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.7/74.7 kB 930.4 kB/s eta 0:00:000:00:01
setup complete — bm25, recency, popularity ready
=== recency-only (check ~0.96) (w_c=0,w_r=1,w_p=0, n=4000) ===
recall@50: 0.9155
recall@100: 0.9460
recall@200: 0.9627

=== popularity-only (check ~0.44) (w_c=0,w_r=0,w_p=1, n=4000) ===
recall@50: 0.1593
recall@100: 0.2630
recall@200: 0.4358

=== recency+popularity (w_c=0,w_r=0.7,w_p=0.3, n=4000) ===
recall@50: 0.9130
recall@100: 0.9543
recall@200: 0.9673

=== content+recency+popularity (w_c=0.2,w_r=0.6,w_p=0.2, n=4000) ===
recall@50: 0.9050
recall@100: 0.9495
recall@200: 0.9617

=== content-heavy (w_c=0.5,w_r=0.3,w_p=0.2, n=4000) ===
recall@50: 0.6240
recall@100: 0.8023
recall@200: 0.8698



In [7]:
!pip install bm25s -q
import re, math, numpy as np, polars as pl, glob, datetime as dt
from bisect import bisect_left
from collections import defaultdict, Counter

DEMO = glob.glob("/kaggle/input/**/ebnerd_demo", recursive=True)[0]
PREFIX="ebnerd"
def _prefix(c): return pl.concat_str([pl.lit(f"{PREFIX}:"), c.cast(pl.Utf8)])
_WORD=re.compile(r"[^\W\d_]+",re.UNICODE)
def tok(t): return _WORD.findall(t.lower()) if t else []

# ---------- parse ----------
a=pl.read_parquet(f"{DEMO}/articles.parquet")
articles=a.select(article_id=_prefix(pl.col("article_id")),
    title=pl.col("title").fill_null(""), abstract=pl.col("subtitle").fill_null(""),
    category_str=pl.col("category_str").fill_null(""), published_time="published_time")
b=pl.read_parquet(f"{DEMO}/train/behaviors.parquet")
imps_full=b.select(user_id=_prefix(pl.col("user_id")), timestamp="impression_time",
    candidate_ids=pl.col("article_ids_inview").list.eval(_prefix(pl.element())),
    labels=pl.col("article_ids_clicked").list.eval(_prefix(pl.element())))
h=pl.read_parquet(f"{DEMO}/train/history.parquet")
hist=h.select(user_id=_prefix(pl.col("user_id")),
    article_ids=pl.col("article_id_fixed").list.eval(_prefix(pl.element())),
    timestamps="impression_time_fixed")

pub_lut={f"{PREFIX}:{aid}":pt for aid,pt in a.select(["article_id","published_time"]).iter_rows()}
cat_lut={aid:c for aid,c in articles.select(["article_id","category_str"]).iter_rows()}
hist_lut={u:(ai or [],ts or []) for u,ai,ts in hist.select(["user_id","article_ids","timestamps"]).iter_rows()}

# ---------- BM25 (content) ----------
ids=[]; corpus=[]
for aid,ti,ab in articles.select(["article_id","title","abstract"]).iter_rows():
    ids.append(aid); corpus.append(tok(f"{ti} {ab}"))
id_to_row={x:i for i,x in enumerate(ids)}
title_lut={aid:tok(ti) for aid,ti in articles.select(["article_id","title"]).iter_rows()}
class BM25:
    def __init__(s,c,k1=1.5,b=0.75):
        s.k1,s.b=k1,b; s.N=len(c); s.tf=[Counter(d) for d in c]
        s.dl=np.array([len(d) for d in c],float); s.avg=s.dl.mean()
        df=Counter()
        for t in s.tf: df.update(t.keys())
        s.idf={w:math.log((s.N-d+.5)/(d+.5)+1) for w,d in df.items()}
    def score(s,q,r):
        if not q: return 0.0
        tf=s.tf[r]; dn=s.k1*(1-s.b+s.b*s.dl[r]/s.avg); v=0.0
        for w in set(q):
            f=tf.get(w,0)
            if f: v+=s.idf.get(w,0)*(f*(s.k1+1))/(f+dn)
        return v
bm25=BM25(corpus)
def hist_q(uid,ts,mh=30):
    ai,tss=hist_lut.get(uid,([],[]))
    if not ai: return []
    if ts and tss and len(tss)==len(ai):
        k=[(x,t) for x,t in zip(ai,tss) if t is not None and t<ts]; k.sort(key=lambda z:z[1]); ai=[x for x,_ in k][-mh:]
    else: ai=ai[-mh:]
    q=[]; [q.extend(title_lut.get(x,[])) for x in ai]; return q

# ---------- recency + popularity ----------
def recency_score(pub,imp_t,tau=6.0):
    if pub is None or imp_t is None: return 0.0
    dh=(imp_t-pub).total_seconds()/3600.0
    return float(np.exp(-dh/tau)) if dh>=0 else 0.0
class PopIdx:
    def __init__(s,ev):
        s.by=defaultdict(list)
        for aid,t in ev:
            if t is not None: s.by[aid].append(t)
        for aid in s.by: s.by[aid].sort()
    def total_before(s,aid,T):
        t=s.by.get(aid); return bisect_left(t,T) if t else 0
click_events=[]
for imp in imps_full.iter_rows(named=True):
    for aid in (imp["labels"] or []): click_events.append((aid,imp["timestamp"]))
popidx=PopIdx(click_events)

# popularity for head/tail slice (global is fine for slicing, not for features)
pop_global=Counter()
for imp in imps_full.iter_rows(named=True):
    for x in (imp["labels"] or []): pop_global[x]+=1
sbp=sorted(pop_global.items(),key=lambda kv:-kv[1]); n_head=max(1,int(0.2*len(sbp)))
head_set={aid for aid,_ in sbp[:n_head]}

# ---------- metrics ----------
def auc_i(sc,lb):
    p=lb==1;n=lb==0;np_,nn=p.sum(),n.sum()
    if np_==0 or nn==0: return None
    o=np.argsort(sc);r=np.empty_like(o,float);r[o]=np.arange(1,len(sc)+1)
    return float((r[p].sum()-np_*(np_+1)/2)/(np_*nn))
def mrr_i(sc,lb):
    rl=lb[np.argsort(-sc)];idx=np.nonzero(rl==1)[0];return 1/(idx[0]+1) if len(idx) else 0.0
def dcg(r): return np.sum(r/np.log2(np.arange(2,len(r)+2)))
def ndcg_i(sc,lb,k):
    rk=lb[np.argsort(-sc)][:k];il=np.sort(lb)[::-1][:k];i=dcg(il);return float(dcg(rk)/i) if i>0 else 0.0
def mm(x):
    lo,hi=x.min(),x.max()
    return np.zeros_like(x) if hi-lo<1e-12 else (x-lo)/(hi-lo)

# ---------- per-candidate signal vectors for one impression ----------
def signals(cand,uid,T):
    q=hist_q(uid,T)
    content=np.array([bm25.score(q,id_to_row[c]) if c in id_to_row else 0.0 for c in cand])
    recency=np.array([recency_score(pub_lut.get(c),T) for c in cand])
    popul =np.array([popidx.total_before(c,T) for c in cand],float)
    return content,recency,popul

# ---------- rerankers ----------
def make_scorer(kind,w=None):
    def f(cand,imp):
        c,r,p=signals(cand,imp["user_id"],imp["timestamp"])
        if kind=="content": return c
        if kind=="recency": return r
        if kind=="popularity": return p
        if kind=="fusion":
            return w[0]*mm(c)+w[1]*mm(r)+w[2]*mm(p)
    return f

SCORERS={
    "content":    make_scorer("content"),
    "recency":    make_scorer("recency"),
    "popularity": make_scorer("popularity"),
    "fusion .4/.3/.3": make_scorer("fusion",(0.4,0.3,0.3)),
    "fusion .2/.5/.3": make_scorer("fusion",(0.2,0.5,0.3)),
}

# ---------- run through harness ----------
mkeys=["auc","mrr","ndcg@5","ndcg@10"]
def agg(sub,nb=500,seed=0):
    rng=np.random.default_rng(seed);o={}
    for m in mkeys:
        v=np.array([r[m] for r in sub if r.get(m) is not None],float)
        if len(v)==0: o[m]=(np.nan,np.nan,np.nan); continue
        bt=np.array([rng.choice(v,len(v),replace=True).mean() for _ in range(nb)])
        o[m]=(v.mean(),*np.percentile(bt,[2.5,97.5]))
    return o

def run(scorer):
    rows=[]
    for imp in imps_full.iter_rows(named=True):
        labs=set(imp["labels"] or []); cand=imp["candidate_ids"]
        if not labs or not cand: continue
        y=np.array([1 if c in labs else 0 for c in cand])
        if y.sum()==0: continue
        sc=np.asarray(scorer(cand,imp),float)
        d={"auc":auc_i(sc,y),"mrr":mrr_i(sc,y),
           "ndcg@5":ndcg_i(sc,y,5),"ndcg@10":ndcg_i(sc,y,10),
           "_slice":"head" if (labs&head_set) else "tail"}
        rows.append(d)
    return rows

print(f"{'method':16s} {'slice':5s} {'AUC':>18s} {'MRR':>8s} {'nDCG@5':>8s} {'nDCG@10':>8s}")
for name,scorer in SCORERS.items():
    rows=run(scorer)
    for sl,sub in [("all",rows),("head",[r for r in rows if r['_slice']=='head']),
                   ("tail",[r for r in rows if r['_slice']=='tail'])]:
        A=agg(sub)
        auc=f"{A['auc'][0]:.3f}[{A['auc'][1]:.3f},{A['auc'][2]:.3f}]"
        print(f"{name:16s} {sl:5s} {auc:>18s} {A['mrr'][0]:>8.3f} {A['ndcg@5'][0]:>8.3f} {A['ndcg@10'][0]:>8.3f}")
    print()

method           slice                AUC      MRR   nDCG@5  nDCG@10
content          all   0.519[0.515,0.523]    0.339    0.373    0.456
content          head  0.531[0.525,0.537]    0.349    0.385    0.466
content          tail  0.508[0.502,0.513]    0.330    0.363    0.447

recency          all   0.503[0.499,0.507]    0.311    0.341    0.427
recency          head  0.451[0.445,0.455]    0.275    0.298    0.394
recency          tail  0.552[0.547,0.557]    0.345    0.382    0.459

popularity       all   0.693[0.689,0.696]    0.449    0.511    0.564
popularity       head  0.825[0.821,0.829]    0.588    0.662    0.687
popularity       tail  0.568[0.563,0.572]    0.316    0.368    0.447

fusion .4/.3/.3  all   0.632[0.628,0.636]    0.419    0.469    0.530
fusion .4/.3/.3  head  0.697[0.691,0.702]    0.477    0.533    0.583
fusion .4/.3/.3  tail  0.570[0.564,0.575]    0.365    0.409    0.479

fusion .2/.5/.3  all   0.610[0.607,0.615]    0.402    0.446    0.511
fusion .2/.5/.3  head  0.653[0

In [8]:
!pip install lightgbm scikit-learn -q
import numpy as np, lightgbm as lgb
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
# assumes from prior cell: imps_full, bm25, hist_q, id_to_row, recency_score,
#   pub_lut, popidx, cat_lut, hist_lut, head_set, auc_i, agg  (re-run reranker cell if kernel reset)

def mm(x):
    lo,hi=x.min(),x.max()
    return np.zeros_like(x) if hi-lo<1e-12 else (x-lo)/(hi-lo)

# ---- user category profile (for category_match feature), leakage-guarded ----
def user_cats(uid,T,mh=30):
    ai,tss=hist_lut.get(uid,([],[]))
    if not ai: return set()
    if T and tss and len(tss)==len(ai):
        keep=[x for x,t in zip(ai,tss) if t is not None and t<T]; ai=keep[-mh:]
    else: ai=ai[-mh:]
    return {cat_lut.get(x) for x in ai if cat_lut.get(x)}

# ---- build feature rows per (impression, candidate) with temporal order ----
def build_features(imps):
    rows=[]
    for imp in imps.iter_rows(named=True):
        labs=set(imp["labels"] or []); cand=imp["candidate_ids"]; T=imp["timestamp"]
        if not labs or not cand: continue
        y=np.array([1 if c in labs else 0 for c in cand])
        if y.sum()==0: continue
        q=hist_q(imp["user_id"],T)
        content=np.array([bm25.score(q,id_to_row[c]) if c in id_to_row else 0.0 for c in cand])
        recency=np.array([recency_score(pub_lut.get(c),T) for c in cand])
        popul=np.array([popidx.total_before(c,T) for c in cand],float)
        ucats=user_cats(imp["user_id"],T)
        catm=np.array([1.0 if cat_lut.get(c) in ucats else 0.0 for c in cand])
        # normalize within slate
        feats=np.column_stack([mm(content),mm(recency),mm(popul),catm])
        rows.append((feats,y,T,("head" if (labs&head_set) else "tail")))
    return rows

all_rows=build_features(imps_full)
all_rows.sort(key=lambda r:r[2])                 # sort by impression time
cut=int(0.8*len(all_rows))
train_rows, test_rows = all_rows[:cut], all_rows[cut:]
print(f"train slates: {len(train_rows)} | test slates: {len(test_rows)}")

def stack(rows):
    X=np.vstack([r[0] for r in rows]); y=np.concatenate([r[1] for r in rows])
    groups=[len(r[1]) for r in rows]
    return X,y,groups
Xtr,ytr,gtr=stack(train_rows); Xte,yte,gte=stack(test_rows)
FEAT=["content","recency","popularity","cat_match"]

# ---- Rung 2: logistic regression ----
sc=StandardScaler().fit(Xtr)
lr=LogisticRegression(max_iter=1000).fit(sc.transform(Xtr),ytr)
print("\n[Logistic] standardized coefficients:")
for f,c in zip(FEAT,lr.coef_[0]): print(f"  {f:12s} {c:+.3f}")
lr_scores=lr.decision_function(sc.transform(Xte))

# ---- Rung 3: LightGBM LambdaMART ----
rk=lgb.LGBMRanker(objective="lambdarank",n_estimators=200,learning_rate=0.05,
                  num_leaves=15,min_child_samples=50,verbose=-1)
rk.fit(Xtr,ytr,group=gtr)
print("\n[LightGBM] feature importances:")
for f,i in zip(FEAT,rk.feature_importances_): print(f"  {f:12s} {i}")
lgb_scores=rk.predict(Xte)

# ---- evaluate both on test slates (per-slate AUC) ----
def eval_scores(scores,rows):
    out={"all":[],"head":[],"tail":[]}; pos=0
    for feats,y,T,sl in rows:
        m=len(y); s=scores[pos:pos+m]; pos+=m
        au=auc_i(s,y)
        if au is not None:
            out["all"].append(au); out[sl].append(au)
    return {k:(np.mean(v) if v else float('nan')) for k,v in out.items()}

print("\n=== TEST-SET AUC (temporal holdout) ===")
print(f"{'model':12s} {'all':>7s} {'head':>7s} {'tail':>7s}")
for name,sc_ in [("logistic",lr_scores),("lightgbm",lgb_scores)]:
    r=eval_scores(sc_,test_rows)
    print(f"{name:12s} {r['all']:>7.3f} {r['head']:>7.3f} {r['tail']:>7.3f}")

# single-signal baselines on the SAME test slates
for fi,fname in enumerate(FEAT):
    r=eval_scores(Xte[:,fi],test_rows)
    print(f"{fname:12s} {r['all']:>7.3f} {r['head']:>7.3f} {r['tail']:>7.3f}")

train slates: 19779 | test slates: 4945

[Logistic] standardized coefficients:
  content      +0.130
  recency      +0.378
  popularity   +0.692
  cat_match    +0.142

[LightGBM] feature importances:
  content      742
  recency      1112
  popularity   845
  cat_match    101

=== TEST-SET AUC (temporal holdout) ===
model            all    head    tail


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRanker was fitted with feature names
  warnings.warn(


logistic       0.688   0.840   0.595
lightgbm       0.717   0.827   0.649
content        0.505   0.524   0.494
recency        0.517   0.446   0.560
popularity     0.679   0.842   0.579
cat_match      0.534   0.522   0.541


In [9]:
!pip install lightgbm sentence-transformers -q
import numpy as np, datetime as dt, lightgbm as lgb
from bisect import bisect_left
from collections import defaultdict
from sentence_transformers import SentenceTransformer

# ============ 1. VERIFY inview order is meaningful (position feature) ============
# If the clicked article's average position is NOT uniform, order carries signal.
pos_of_click=[]
for imp in imps_full.iter_rows(named=True):
    labs=set(imp["labels"] or []); cand=imp["candidate_ids"]
    if not labs or not cand: continue
    for i,c in enumerate(cand):
        if c in labs: pos_of_click.append(i/max(1,len(cand)-1))  # normalized 0..1
print(f"clicked normalized position: mean={np.mean(pos_of_click):.3f} (0.5=uniform/no signal)")

# ============ 2. leakage-safe CTR + multi-window popularity ============
# build click AND impression events (both before T) per article
click_ev=defaultdict(list); imp_ev=defaultdict(list)
for imp in imps_full.iter_rows(named=True):
    T=imp["timestamp"]
    for c in (imp["candidate_ids"] or []): imp_ev[c].append(T)      # shown
    for c in (imp["labels"] or []):        click_ev[c].append(T)    # clicked
for d in (click_ev,imp_ev):
    for k in d: d[k].sort()

def cnt_before(d,aid,T,win_h=None):
    t=d.get(aid)
    if not t: return 0
    hi=bisect_left(t,T)
    if win_h is None: return hi
    lo=bisect_left(t,T-dt.timedelta(hours=win_h))
    return hi-lo

def ctr_before(aid,T,win_h=None):
    c=cnt_before(click_ev,aid,T,win_h); s=cnt_before(imp_ev,aid,T,win_h)
    return c/s if s>0 else 0.0

# ============ 3. E5 embeddings (for cosine + best-match features) ============
DEMO_A = pl.read_parquet(f"{glob.glob('/kaggle/input/**/ebnerd_demo',recursive=True)[0]}/articles.parquet")
e5=SentenceTransformer("intfloat/multilingual-e5-base")
e5_ids=[f"{PREFIX}:{i}" for i in DEMO_A["article_id"].to_list()]
e5_txt=[f"passage: {ti or ''} {ab or ''}".strip() for ti,ab in
        zip(DEMO_A["title"].to_list(), DEMO_A["subtitle"].to_list())]
e5_mat=e5.encode(e5_txt,batch_size=256,normalize_embeddings=True,convert_to_numpy=True,show_progress_bar=True)
e5_by_id={aid:e5_mat[i] for i,aid in enumerate(e5_ids)}

def hist_vecs(uid,T,mh=30):
    ai,tss=hist_lut.get(uid,([],[]))
    if not ai: return []
    if T and tss and len(tss)==len(ai):
        keep=[x for x,t in zip(ai,tss) if t is not None and t<T]; ai=keep[-mh:]
    else: ai=ai[-mh:]
    return [e5_by_id[x] for x in ai if x in e5_by_id]

# ============ 4. build enriched feature matrix ============
def recency_score(pub,T,tau=6.0):
    if pub is None or T is None: return 0.0
    dh=(T-pub).total_seconds()/3600.0
    return float(np.exp(-dh/tau)) if dh>=0 else 0.0
def mm(x):
    lo,hi=x.min(),x.max(); return np.zeros_like(x) if hi-lo<1e-12 else (x-lo)/(hi-lo)

def user_cats(uid,T,mh=30):
    ai,tss=hist_lut.get(uid,([],[]))
    if not ai: return {}
    if T and tss and len(tss)==len(ai):
        keep=[x for x,t in zip(ai,tss) if t is not None and t<T]; ai=keep[-mh:]
    else: ai=ai[-mh:]
    cats=[cat_lut.get(x) for x in ai]
    tot=len([c for c in cats if c]); 
    from collections import Counter as C
    cc=C(c for c in cats if c)
    return {k:v/tot for k,v in cc.items()} if tot else {}

FEAT=["content_bm25","e5_mean","e5_bestmatch","recency",
      "pop_1h","pop_24h","pop_7d","pop_total","pop_velocity",
      "ctr_24h","ctr_total","cat_affinity","position","slate_size"]

def build(imps):
    rows=[]
    for imp in imps.iter_rows(named=True):
        labs=set(imp["labels"] or []); cand=imp["candidate_ids"]; T=imp["timestamp"]
        if not labs or not cand: continue
        y=np.array([1 if c in labs else 0 for c in cand])
        if y.sum()==0: continue
        m=len(cand)
        q=hist_q(imp["user_id"],T)
        hv=hist_vecs(imp["user_id"],T); umean=np.mean(hv,0) if hv else None
        ucat=user_cats(imp["user_id"],T)
        F=[]
        for i,c in enumerate(cand):
            bm=bm25.score(q,id_to_row[c]) if c in id_to_row else 0.0
            cv=e5_by_id.get(c)
            e5m=float(umean@cv) if (umean is not None and cv is not None) else 0.0
            e5b=float(max((v@cv for v in hv),default=0.0)) if cv is not None else 0.0
            rec=recency_score(pub_lut.get(c),T)
            p1=cnt_before(click_ev,c,T,1); p24=cnt_before(click_ev,c,T,24)
            p7=cnt_before(click_ev,c,T,24*7); ptot=cnt_before(click_ev,c,T)
            pvel=p1/(p24+1)
            c24=ctr_before(c,T,24); ctot=ctr_before(c,T)
            caff=ucat.get(cat_lut.get(c),0.0)
            pos=i/max(1,m-1)
            F.append([bm,e5m,e5b,rec,p1,p24,p7,ptot,pvel,c24,ctot,caff,pos,m])
        F=np.array(F)
        # normalize the continuous cols within slate (not position/slate_size/ctr)
        for col in [0,1,2,3,4,5,6,7,8]:
            F[:,col]=mm(F[:,col])
        rows.append((F,y,T,("head" if (labs&head_set) else "tail")))
    return rows

all_rows=build(imps_full); all_rows.sort(key=lambda r:r[2])
cut=int(0.8*len(all_rows)); tr,te=all_rows[:cut],all_rows[cut:]
def stack(rows):
    return (np.vstack([r[0] for r in rows]),
            np.concatenate([r[1] for r in rows]),
            [len(r[1]) for r in rows])
Xtr,ytr,gtr=stack(tr); Xte,yte,gte=stack(te)
print(f"train {len(tr)} / test {len(te)} slates, {len(FEAT)} features")

# ============ 5. train LightGBM ============
rk=lgb.LGBMRanker(objective="lambdarank",n_estimators=400,learning_rate=0.03,
                  num_leaves=31,min_child_samples=50,importance_type="gain",verbose=-1)
rk.fit(Xtr,ytr,group=gtr)
sc=rk.predict(Xte)

def auc_i(s,lb):
    p=lb==1;n=lb==0;np_,nn=p.sum(),n.sum()
    if np_==0 or nn==0: return None
    o=np.argsort(s);r=np.empty_like(o,float);r[o]=np.arange(1,len(s)+1)
    return float((r[p].sum()-np_*(np_+1)/2)/(np_*nn))
def ev(scores,rows):
    out={"all":[],"head":[],"tail":[]};pos=0
    for F,y,T,sl in rows:
        m=len(y);a=auc_i(scores[pos:pos+m],y);pos+=m
        if a is not None: out["all"].append(a);out[sl].append(a)
    return {k:np.mean(v) for k,v in out.items()}

r=ev(sc,te)
print(f"\n=== ENRICHED LightGBM  AUC: all={r['all']:.3f}  head={r['head']:.3f}  tail={r['tail']:.3f} ===")
print("(prev best: all=0.717 head=0.827 tail=0.649)\n")
print("feature importances (gain):")
for f,i in sorted(zip(FEAT,rk.feature_importances_),key=lambda z:-z[1]):
    print(f"  {f:14s} {i:>10.0f}")

clicked normalized position: mean=0.504 (0.5=uniform/no signal)


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Batches:   0%|          | 0/47 [00:00<?, ?it/s]

train 19779 / test 4945 slates, 14 features


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRanker was fitted with feature names
  warnings.warn(



=== ENRICHED LightGBM  AUC: all=0.756  head=0.826  tail=0.714 ===
(prev best: all=0.717 head=0.827 tail=0.649)

feature importances (gain):
  pop_24h            111421
  recency             53172
  slate_size          36962
  cat_affinity        29481
  ctr_24h             28524
  pop_1h              25519
  pop_velocity        22929
  ctr_total           21274
  content_bm25        15122
  e5_mean             13190
  e5_bestmatch        11317
  position             9949
  pop_7d               9312
  pop_total               0


In [12]:
import polars as pl, glob
DEMO = glob.glob("/kaggle/input/**/ebnerd_demo", recursive=True)[0]
b = pl.read_parquet(f"{DEMO}/train/behaviors.parquet")
imps_full = b.select(
    user_id=_prefix(pl.col("user_id")), timestamp="impression_time",
    session_id="session_id",                                   # ← the missing column
    candidate_ids=pl.col("article_ids_inview").list.eval(_prefix(pl.element())),
    labels=pl.col("article_ids_clicked").list.eval(_prefix(pl.element())),
)
print("session_id present:", "session_id" in imps_full.columns)

session_id present: True


In [13]:
import numpy as np, datetime as dt, lightgbm as lgb
from collections import defaultdict

# ============ session structure check ============
sess_imps=defaultdict(list)
for imp in imps_full.iter_rows(named=True):
    sess_imps[imp["session_id"]].append((imp["timestamp"], imp))
sizes=[len(v) for v in sess_imps.values()]
print(f"sessions: {len(sess_imps)} | impressions/session: mean={np.mean(sizes):.2f}, "
      f"median={int(np.median(sizes))}, max={max(sizes)}")
print(f"sessions with >1 impression: {sum(1 for s in sizes if s>1)/len(sizes):.1%}")

# ============ build per-session click history (leakage-safe within session) ============
# for each impression, gather categories the user clicked EARLIER in the same session
# (strictly before this impression's timestamp)
sess_click_cats=defaultdict(list)   # session_id -> list of (T, clicked_category)
for imp in imps_full.iter_rows(named=True):
    T=imp["timestamp"]; sid=imp["session_id"]
    for c in (imp["labels"] or []):
        cat=cat_lut.get(c)
        if cat: sess_click_cats[sid].append((T,cat))
for sid in sess_click_cats: sess_click_cats[sid].sort()

def session_cat_match(sid,T,candidate_cat):
    """1 if candidate's category was clicked earlier in this session, else 0.
       count of prior in-session clicks in that category (leakage: strictly before T)."""
    events=sess_click_cats.get(sid,[])
    if not events or candidate_cat is None: return 0.0, 0
    prior=[cat for (t,cat) in events if t<T]
    if not prior: return 0.0, 0
    match=sum(1 for cat in prior if cat==candidate_cat)
    return (1.0 if match>0 else 0.0), match  # binary match, and count

# ============ rebuild features WITH session signals ============
FEAT2=["content_bm25","e5_mean","e5_bestmatch","recency",
       "pop_1h","pop_24h","pop_7d","pop_velocity",
       "ctr_24h","ctr_total","cat_affinity","position","slate_size",
       "sess_cat_match","sess_cat_count","sess_depth"]

def recency_score(pub,T,tau=6.0):
    if pub is None or T is None: return 0.0
    dh=(T-pub).total_seconds()/3600.0
    return float(np.exp(-dh/tau)) if dh>=0 else 0.0
def mm(x):
    lo,hi=x.min(),x.max(); return np.zeros_like(x) if hi-lo<1e-12 else (x-lo)/(hi-lo)

def build2(imps):
    rows=[]
    for imp in imps.iter_rows(named=True):
        labs=set(imp["labels"] or []); cand=imp["candidate_ids"]; T=imp["timestamp"]; sid=imp["session_id"]
        if not labs or not cand: continue
        y=np.array([1 if c in labs else 0 for c in cand])
        if y.sum()==0: continue
        m=len(cand)
        q=hist_q(imp["user_id"],T)
        hv=hist_vecs(imp["user_id"],T); umean=np.mean(hv,0) if hv else None
        ucat=user_cats(imp["user_id"],T)
        # session depth = how many in-session clicks happened before this impression
        sdepth=len([1 for (t,_) in sess_click_cats.get(sid,[]) if t<T])
        F=[]
        for i,c in enumerate(cand):
            bm=bm25.score(q,id_to_row[c]) if c in id_to_row else 0.0
            cv=e5_by_id.get(c)
            e5m=float(umean@cv) if (umean is not None and cv is not None) else 0.0
            e5b=float(max((v@cv for v in hv),default=0.0)) if cv is not None else 0.0
            rec=recency_score(pub_lut.get(c),T)
            p1=cnt_before(click_ev,c,T,1); p24=cnt_before(click_ev,c,T,24)
            p7=cnt_before(click_ev,c,T,24*7); pvel=p1/(p24+1)
            c24=ctr_before(c,T,24); ctot=ctr_before(c,T)
            caff=ucat.get(cat_lut.get(c),0.0); pos=i/max(1,m-1)
            smatch,scount=session_cat_match(sid,T,cat_lut.get(c))
            F.append([bm,e5m,e5b,rec,p1,p24,p7,pvel,c24,ctot,caff,pos,m,smatch,scount,sdepth])
        F=np.array(F)
        for col in [0,1,2,3,4,5,6,7]: F[:,col]=mm(F[:,col])
        rows.append((F,y,T,("head" if (labs&head_set) else "tail")))
    return rows

rows2=build2(imps_full); rows2.sort(key=lambda r:r[2])
cut=int(0.8*len(rows2)); tr,te=rows2[:cut],rows2[cut:]
def stack(rows): return (np.vstack([r[0] for r in rows]),
                         np.concatenate([r[1] for r in rows]),[len(r[1]) for r in rows])
Xtr,ytr,gtr=stack(tr); Xte,yte,gte=stack(te)

rk=lgb.LGBMRanker(objective="lambdarank",n_estimators=400,learning_rate=0.03,
                  num_leaves=31,min_child_samples=50,importance_type="gain",verbose=-1)
rk.fit(Xtr,ytr,group=gtr); sc=rk.predict(Xte)

def auc_i(s,lb):
    p=lb==1;n=lb==0;np_,nn=p.sum(),n.sum()
    if np_==0 or nn==0: return None
    o=np.argsort(s);r=np.empty_like(o,float);r[o]=np.arange(1,len(s)+1)
    return float((r[p].sum()-np_*(np_+1)/2)/(np_*nn))
def ev(scores,rows):
    out={"all":[],"head":[],"tail":[]};pos=0
    for F,y,T,sl in rows:
        m=len(y);a=auc_i(scores[pos:pos+m],y);pos+=m
        if a is not None: out["all"].append(a);out[sl].append(a)
    return {k:np.mean(v) for k,v in out.items()}
r=ev(sc,te)
print(f"\n=== +SESSION LightGBM  AUC: all={r['all']:.3f} head={r['head']:.3f} tail={r['tail']:.3f} ===")
print("(prev: all=0.756 head=0.826 tail=0.714)\n")
for f,i in sorted(zip(FEAT2,rk.feature_importances_),key=lambda z:-z[1]):
    print(f"  {f:16s} {i:>10.0f}")

sessions: 12944 | impressions/session: mean=1.91, median=1, max=21
sessions with >1 impression: 44.4%


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRanker was fitted with feature names
  warnings.warn(



=== +SESSION LightGBM  AUC: all=0.757 head=0.827 tail=0.715 ===
(prev: all=0.756 head=0.826 tail=0.714)

  pop_24h              110832
  recency               54307
  slate_size            37398
  cat_affinity          29625
  pop_1h                26192
  ctr_24h               25686
  ctr_total             23055
  pop_velocity          22105
  content_bm25          14721
  e5_bestmatch          12113
  e5_mean               11403
  position               9183
  pop_7d                 9110
  sess_depth             3984
  sess_cat_match          783
  sess_cat_count          640


In [14]:
import polars as pl, glob
SMALL = glob.glob("/kaggle/input/**/ebnerd_small", recursive=True)[0]
print("small root:", SMALL)

a = pl.read_parquet(f"{SMALL}/articles.parquet")
btr = pl.read_parquet(f"{SMALL}/train/behaviors.parquet")
htr = pl.read_parquet(f"{SMALL}/train/history.parquet")
bva = pl.read_parquet(f"{SMALL}/validation/behaviors.parquet")

print(f"articles:        {a.height:,}")
print(f"train impressions:   {btr.height:,}")
print(f"train users(history):{htr.height:,}")
print(f"val impressions:     {bva.height:,}")

# quick sanity: same schema as demo?
print("\narticles cols match demo:", set(a.columns) >= {"article_id","title","subtitle","body","category_str","published_time"})
print("behaviors cols match demo:", set(btr.columns) >= {"impression_id","article_ids_inview","article_ids_clicked","user_id","impression_time","session_id"})

small root: /kaggle/input/datasets/wrathofgod123/ebnerd/ebnerd_small
articles:        20,738
train impressions:   232,887
train users(history):15,143
val impressions:     244,647

articles cols match demo: True
behaviors cols match demo: True


In [15]:
!pip install lightgbm sentence-transformers bm25s -q
import re, math, numpy as np, polars as pl, glob, datetime as dt, lightgbm as lgb
from bisect import bisect_left
from collections import defaultdict, Counter
from sentence_transformers import SentenceTransformer

SMALL = glob.glob("/kaggle/input/**/ebnerd_small", recursive=True)[0]
W2V  = "/kaggle/input/datasets/wrathofgod123/ebnerd/Ekstra_Bladet_word2vec/Ekstra_Bladet_word2vec/document_vector.parquet"
PREFIX="ebnerd"
def _prefix(c): return pl.concat_str([pl.lit(f"{PREFIX}:"), c.cast(pl.Utf8)])
_WORD=re.compile(r"[^\W\d_]+",re.UNICODE)
def tok(t): return _WORD.findall(t.lower()) if t else []

# ---- parse train + validation ----
a=pl.read_parquet(f"{SMALL}/articles.parquet")
articles=a.select(article_id=_prefix(pl.col("article_id")),
    title=pl.col("title").fill_null(""), abstract=pl.col("subtitle").fill_null(""),
    category_str=pl.col("category_str").fill_null(""), published_time="published_time")
def parse_beh(split):
    b=pl.read_parquet(f"{SMALL}/{split}/behaviors.parquet")
    return b.select(user_id=_prefix(pl.col("user_id")), timestamp="impression_time",
        session_id="session_id",
        candidate_ids=pl.col("article_ids_inview").list.eval(_prefix(pl.element())),
        labels=pl.col("article_ids_clicked").list.eval(_prefix(pl.element())))
imps_tr=parse_beh("train"); imps_va=parse_beh("validation")
def parse_hist(split):
    h=pl.read_parquet(f"{SMALL}/{split}/history.parquet")
    return {u:(ai or [],ts or []) for u,ai,ts in h.select(
        user_id=_prefix(pl.col("user_id")),
        article_ids=pl.col("article_id_fixed").list.eval(_prefix(pl.element())),
        timestamps="impression_time_fixed").iter_rows()}
hist_lut={**parse_hist("train"), **parse_hist("validation")}

pub_lut={f"{PREFIX}:{aid}":pt for aid,pt in a.select(["article_id","published_time"]).iter_rows()}
cat_lut={aid:c for aid,c in articles.select(["article_id","category_str"]).iter_rows()}

# ---- BM25 ----
ids=[]; corpus=[]
for aid,ti,ab in articles.select(["article_id","title","abstract"]).iter_rows():
    ids.append(aid); corpus.append(tok(f"{ti} {ab}"))
id_to_row={x:i for i,x in enumerate(ids)}
title_lut={aid:tok(ti) for aid,ti in articles.select(["article_id","title"]).iter_rows()}
class BM25:
    def __init__(s,c,k1=1.5,b=0.75):
        s.k1,s.b=k1,b; s.N=len(c); s.tf=[Counter(d) for d in c]
        s.dl=np.array([len(d) for d in c],float); s.avg=s.dl.mean()
        df=Counter()
        for t in s.tf: df.update(t.keys())
        s.idf={w:math.log((s.N-d+.5)/(d+.5)+1) for w,d in df.items()}
    def score(s,q,r):
        if not q: return 0.0
        tf=s.tf[r]; dn=s.k1*(1-s.b+s.b*s.dl[r]/s.avg); v=0.0
        for w in set(q):
            f=tf.get(w,0)
            if f: v+=s.idf.get(w,0)*(f*(s.k1+1))/(f+dn)
        return v
bm25=BM25(corpus)
def hist_q(uid,ts,mh=30):
    ai,tss=hist_lut.get(uid,([],[]))
    if not ai: return []
    if ts and tss and len(tss)==len(ai):
        k=[(x,t) for x,t in zip(ai,tss) if t is not None and t<ts]; k.sort(key=lambda z:z[1]); ai=[x for x,_ in k][-mh:]
    else: ai=ai[-mh:]
    q=[]; [q.extend(title_lut.get(x,[])) for x in ai]; return q

# ---- popularity + CTR from TRAIN clicks only (leakage: val uses train history) ----
click_ev=defaultdict(list); imp_ev=defaultdict(list)
for imp in imps_tr.iter_rows(named=True):
    T=imp["timestamp"]
    for c in (imp["candidate_ids"] or []): imp_ev[c].append(T)
    for c in (imp["labels"] or []):        click_ev[c].append(T)
for d in (click_ev,imp_ev):
    for k in d: d[k].sort()
def cnt_before(d,aid,T,win_h=None):
    t=d.get(aid)
    if not t: return 0
    hi=bisect_left(t,T)
    if win_h is None: return hi
    return hi-bisect_left(t,T-dt.timedelta(hours=win_h))
def ctr_before(aid,T,win_h=None):
    c=cnt_before(click_ev,aid,T,win_h); s=cnt_before(imp_ev,aid,T,win_h)
    return c/s if s>0 else 0.0

# ---- E5 encode (GPU) ----
e5=SentenceTransformer("intfloat/multilingual-e5-base")
e5_ids=[f"{PREFIX}:{i}" for i in a["article_id"].to_list()]
e5_txt=[f"passage: {ti or ''} {ab or ''}".strip() for ti,ab in zip(a["title"].to_list(),a["subtitle"].to_list())]
e5_mat=e5.encode(e5_txt,batch_size=256,normalize_embeddings=True,convert_to_numpy=True,show_progress_bar=True)
e5_by_id={aid:e5_mat[i] for i,aid in enumerate(e5_ids)}

# head set for slicing (train clicks)
pop_g=Counter()
for imp in imps_tr.iter_rows(named=True):
    for x in (imp["labels"] or []): pop_g[x]+=1
sbp=sorted(pop_g.items(),key=lambda kv:-kv[1]); head_set={aid for aid,_ in sbp[:max(1,int(0.2*len(sbp)))]}
print("setup done. articles:",len(ids),"train imps:",imps_tr.height,"val imps:",imps_va.height)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/82 [00:00<?, ?it/s]

setup done. articles: 20738 train imps: 232887 val imps: 244647


In [16]:
def recency_score(pub,T,tau=6.0):
    if pub is None or T is None: return 0.0
    dh=(T-pub).total_seconds()/3600.0
    return float(np.exp(-dh/tau)) if dh>=0 else 0.0
def mm(x):
    lo,hi=x.min(),x.max(); return np.zeros_like(x) if hi-lo<1e-12 else (x-lo)/(hi-lo)
def user_cats(uid,T,mh=30):
    ai,tss=hist_lut.get(uid,([],[]))
    if not ai: return {}
    if T and tss and len(tss)==len(ai):
        keep=[x for x,t in zip(ai,tss) if t is not None and t<T]; ai=keep[-mh:]
    else: ai=ai[-mh:]
    cats=[cat_lut.get(x) for x in ai]; tot=len([c for c in cats if c])
    cc=Counter(c for c in cats if c)
    return {k:v/tot for k,v in cc.items()} if tot else {}
def hist_vecs(uid,T,mh=30):
    ai,tss=hist_lut.get(uid,([],[]))
    if not ai: return []
    if T and tss and len(tss)==len(ai):
        keep=[x for x,t in zip(ai,tss) if t is not None and t<T]; ai=keep[-mh:]
    else: ai=ai[-mh:]
    return [e5_by_id[x] for x in ai if x in e5_by_id]

FEAT=["content_bm25","e5_mean","e5_bestmatch","recency",
      "pop_1h","pop_24h","pop_7d","pop_total","pop_velocity",
      "ctr_24h","ctr_total","cat_affinity","position","slate_size"]

def build(imps, tag):
    rows=[]; total=imps.height; step=max(1,total//10); k=0
    for imp in imps.iter_rows(named=True):
        k+=1
        if k % step == 0: print(f"  {tag}: {k}/{total}")
        labs=set(imp["labels"] or []); cand=imp["candidate_ids"]; T=imp["timestamp"]
        if not labs or not cand: continue
        y=np.array([1 if c in labs else 0 for c in cand])
        if y.sum()==0: continue
        m=len(cand); q=hist_q(imp["user_id"],T)
        hv=hist_vecs(imp["user_id"],T); umean=np.mean(hv,0) if hv else None
        ucat=user_cats(imp["user_id"],T)
        F=[]
        for i,c in enumerate(cand):
            bm=bm25.score(q,id_to_row[c]) if c in id_to_row else 0.0
            cv=e5_by_id.get(c)
            e5m=float(umean@cv) if (umean is not None and cv is not None) else 0.0
            e5b=float(max((v@cv for v in hv),default=0.0)) if cv is not None else 0.0
            rec=recency_score(pub_lut.get(c),T)
            p1=cnt_before(click_ev,c,T,1); p24=cnt_before(click_ev,c,T,24)
            p7=cnt_before(click_ev,c,T,24*7); ptot=cnt_before(click_ev,c,T)
            pvel=p1/(p24+1); c24=ctr_before(c,T,24); ctot=ctr_before(c,T)
            caff=ucat.get(cat_lut.get(c),0.0); pos=i/max(1,m-1)
            F.append([bm,e5m,e5b,rec,p1,p24,p7,ptot,pvel,c24,ctot,caff,pos,m])
        F=np.array(F)
        for col in [0,1,2,3,4,5,6,7,8]: F[:,col]=mm(F[:,col])
        rows.append((F,y,T,("head" if (labs&head_set) else "tail")))
    return rows

print("building train features...")
tr=build(imps_tr,"train")
print("building val features...")
te=build(imps_va,"val")

def stack(rows): return (np.vstack([r[0] for r in rows]),
                         np.concatenate([r[1] for r in rows]),[len(r[1]) for r in rows])
Xtr,ytr,gtr=stack(tr); Xte,yte,gte=stack(te)
print(f"train {len(tr)} slates / val {len(te)} slates")

rk=lgb.LGBMRanker(objective="lambdarank",n_estimators=400,learning_rate=0.03,
                  num_leaves=31,min_child_samples=50,importance_type="gain",verbose=-1)
rk.fit(Xtr,ytr,group=gtr); sc=rk.predict(Xte)

def auc_i(s,lb):
    p=lb==1;n=lb==0;np_,nn=p.sum(),n.sum()
    if np_==0 or nn==0: return None
    o=np.argsort(s);r=np.empty_like(o,float);r[o]=np.arange(1,len(s)+1)
    return float((r[p].sum()-np_*(np_+1)/2)/(np_*nn))
def ev(scores,rows):
    out={"all":[],"head":[],"tail":[]};pos=0
    for F,y,T,sl in rows:
        m=len(y);a=auc_i(scores[pos:pos+m],y);pos+=m
        if a is not None: out["all"].append(a);out[sl].append(a)
    return {k:np.mean(v) for k,v in out.items()}
r=ev(sc,te)
print(f"\n=== EB-NeRD SMALL — LightGBM AUC: all={r['all']:.3f} head={r['head']:.3f} tail={r['tail']:.3f} ===")
print("(demo was: all=0.756 head=0.826 tail=0.714)\n")
for f,i in sorted(zip(FEAT,rk.feature_importances_),key=lambda z:-z[1]):
    print(f"  {f:14s} {i:>10.0f}")

building train features...
  train: 23288/232887
  train: 46576/232887
  train: 69864/232887
  train: 93152/232887
  train: 116440/232887
  train: 139728/232887
  train: 163016/232887
  train: 186304/232887
  train: 209592/232887
  train: 232880/232887
building val features...
  val: 24464/244647
  val: 48928/244647
  val: 73392/244647
  val: 97856/244647
  val: 122320/244647
  val: 146784/244647
  val: 171248/244647
  val: 195712/244647
  val: 220176/244647
  val: 244640/244647
train 232887 slates / val 244647 slates


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRanker was fitted with feature names
  warnings.warn(



=== EB-NeRD SMALL — LightGBM AUC: all=0.568 head=0.679 tail=0.566 ===
(demo was: all=0.756 head=0.826 tail=0.714)

  pop_24h           1203704
  ctr_24h            840486
  pop_1h             617354
  slate_size         439850
  recency            283887
  e5_bestmatch       270793
  cat_affinity       265553
  e5_mean            107197
  pop_velocity        97661
  ctr_total           94195
  content_bm25        51683
  pop_7d              34771
  position            22065
  pop_total               0


In [19]:
# ---- ROLLING popularity + CTR: events from BOTH train and val, counted strictly before T ----
click_ev=defaultdict(list); imp_ev=defaultdict(list)
for imps in (imps_tr, imps_va):
    for imp in imps.iter_rows(named=True):
        T=imp["timestamp"]
        for c in (imp["candidate_ids"] or []): imp_ev[c].append(T)
        for c in (imp["labels"] or []):        click_ev[c].append(T)
for d in (click_ev,imp_ev):
    for k in d: d[k].sort()
print("rolling event index built. articles with clicks:", len(click_ev))

# cnt_before/ctr_before unchanged — still strictly-before-T (leakage-safe)

# ---- rebuild features with rolling popularity ----
print("building train features (rolling)...")
tr=build(imps_tr,"train")
print("building val features (rolling)...")
te=build(imps_va,"val")

Xtr,ytr,gtr=stack(tr); Xte,yte,gte=stack(te)
print(f"train {len(tr)} slates / val {len(te)} slates")

rk=lgb.LGBMRanker(objective="lambdarank",n_estimators=400,learning_rate=0.03,
                  num_leaves=31,min_child_samples=50,importance_type="gain",verbose=-1)
rk.fit(Xtr,ytr,group=gtr); sc=rk.predict(Xte)

r=ev(sc,te)
print(f"\n=== EB-NeRD SMALL — ROLLING popularity — AUC: all={r['all']:.3f} head={r['head']:.3f} tail={r['tail']:.3f} ===")
print("(frozen train-popularity was: all=0.568 | demo in-period: 0.756)\n")
for f,i in sorted(zip(FEAT,rk.feature_importances_),key=lambda z:-z[1]):
    print(f"  {f:14s} {i:>10.0f}")

rolling event index built. articles with clicks: 3080
building train features (rolling)...
  train: 23288/232887
  train: 46576/232887
  train: 69864/232887
  train: 93152/232887
  train: 116440/232887
  train: 139728/232887
  train: 163016/232887
  train: 186304/232887
  train: 209592/232887
  train: 232880/232887
building val features (rolling)...
  val: 24464/244647
  val: 48928/244647
  val: 73392/244647
  val: 97856/244647
  val: 122320/244647
  val: 146784/244647
  val: 171248/244647
  val: 195712/244647
  val: 220176/244647
  val: 244640/244647
train 232887 slates / val 244647 slates


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRanker was fitted with feature names
  warnings.warn(



=== EB-NeRD SMALL — ROLLING popularity — AUC: all=0.773 head=0.664 tail=0.775 ===
(frozen train-popularity was: all=0.568 | demo in-period: 0.756)

  pop_24h           1203704
  ctr_24h            840486
  pop_1h             617354
  slate_size         439850
  recency            283887
  e5_bestmatch       270793
  cat_affinity       265553
  e5_mean            107197
  pop_velocity        97661
  ctr_total           94195
  content_bm25        51683
  pop_7d              34771
  position            22065
  pop_total               0
